# V3-H2 — Exact Poker Semantics + Counterfactual Action Residuals

**Score-seeking experiment built directly from the verified V2.2 source.**

V2.2 remains the base pair-risk / behavior / three-specialist evidence pipeline.
H2 adds one conceptual change only: **retrospective poker semantics** for the pair
and hand-ranking layers.

H2 computes an exact best-five-card Hold'em ordering from each member's two hole
cards plus the completed board for selected development/evaluation pair-hands.
It then combines exact hand strength with per-player normal-action baselines to
create counterfactual-style residual features such as:

- strong-hand passivity against the partner,
- chip transfer despite a large private-strength mismatch,
- pair aggression above each player's normal baseline followed by outsider folds,
- strength-weighted heads-up checking/calling,
- exact-strength-aware directed / soft-play / isolation hand scores.

The existing V2.2 behavior classifier is deliberately unchanged.

Cost control:
- exact semantics are built for all **labeled development pairs**;
- evaluation semantics are built only for the top base-risk candidate tail;
- H2 pair and evidence models are small second-stage models;
- if validation does not improve, H2 automatically falls back to V2.2 outputs.

The final notebook writes `/kaggle/working/submission.csv` plus a detailed H2
validation summary. **Do not submit before reviewing that summary.**


In [1]:
# ============================================================
# 0. IMPORTS / STRICT CONFIG
# ============================================================

import os
import gc
import json
import math
import shutil
import warnings
import importlib.util
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import polars as pl
import duckdb

from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from xgboost import XGBClassifier, XGBRanker

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 240)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(100)

SEED = 42
N_FOLDS = 5

RUN_ROBUST_GROUP_CV = False
RUN_TEMPORAL_DIAGNOSTIC = True

# Behavior routing is nested-tuned inside each outer fold.
# The grid is predeclared before the strict OOF run; no outer-validation label
# participates in selecting the rate applied to that outer fold.
BEHAVIOR_ACTIVATION_MODE = 'risk'
BEHAVIOR_RATE_GRID = np.array([
    0.001,0.0015,0.0025,0.004,0.006,0.008,0.010,
    0.015,0.020,0.030,0.050,0.075,0.100
], dtype=np.float64)
BEHAVIOR_RATE_TUNING_OBJECTIVE = 'pu_weighted_stable_behavior_map'

# Fixed before this strict OOF evaluation: do not select it on the same OOF labels.
EVIDENCE_MODEL_WEIGHT = 0.70
EVIDENCE_MODEL_PAIR_LIMIT = 25_000
EVIDENCE_BATCH_SIZE = 250_000

# ---------------- H2 exact-semantics experiment ----------------
H2_EVAL_PAIR_LIMIT = 35_000
H2_PAIR_BLEND_WEIGHT = 0.25
H2_EVIDENCE_BLEND_WEIGHT = 0.35
H2_MIN_PAIR_AP_GAIN = 0.0015
H2_MIN_EVIDENCE_MAP_GAIN = 0.005
H2_MIN_COMPOSITE_GAIN = 0.0015
H2_PAIR_N_ESTIMATORS = 260
H2_EVIDENCE_N_ESTIMATORS = 180
H2_SEMANTIC_BATCH_SIZE = 250_000

H2_NUMBA_PRECHECK = importlib.util.find_spec('numba') is not None
assert H2_NUMBA_PRECHECK, 'H2 requires numba; aborting before expensive feature/training work.'

# Historical V2.2 executed anchor, reporting only.
V22_PUBLIC_LB_ANCHOR = 0.66223
V22_LOCAL_PAIR_AP_ANCHOR = 0.9787889233638924
V22_LOCAL_BEHAVIOR_MAP_ANCHOR = 0.9238589474890628
V22_LOCAL_EVIDENCE_MAP_ANCHOR = 0.35145683990442056
V22_LOCAL_COMPOSITE_ANCHOR = 0.8478295090845152

SHRINKAGE_ALPHA = 20.0

# Label-independent development unknown sampling.
# V2a used 60 generic + 12 label-informed controls; strict V2.1 samples the same
# total count without looking at disclosed positive players. Fold-specific training
# weights can still downweight unknown pairs touching OUTER-TRAIN positive players.
UNKNOWN_PER_TABLE = 72
UNKNOWN_BASE_WEIGHT = 0.22
UNKNOWN_POSITIVE_PLAYER_WEIGHT = 0.12
POSITIVE_FIT_WEIGHT = 2.0

# Strict build: rebuild derived caches on every Run All so edited formulas cannot
# silently reuse stale intermediate parquet files.
PIPELINE_BUILD_ID = 'v3_h2_exact_semantics_20260908'
CACHE_VERSION = f'poker_{PIPELINE_BUILD_ID}'
RESET_CACHE = True

DATA_DIR = Path('/kaggle/input/competitions/detect-suspicious-value-transfers-in-poker')
WORK_DIR = Path('/kaggle/working') / CACHE_VERSION

if RESET_CACHE and WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

PLAYERS_PATH = DATA_DIR / 'players.parquet'
HANDS_PATH = DATA_DIR / 'hands.parquet'
SEATS_PATH = DATA_DIR / 'seats.parquet'
ACTIONS_PATH = DATA_DIR / 'actions.parquet'
DEV_LABELS_PATH = DATA_DIR / 'development_labels.csv'
DEV_EVIDENCE_PATH = DATA_DIR / 'development_evidence.csv'
EVAL_PAIRS_PATH = DATA_DIR / 'evaluation_pairs.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'sample_submission.csv'

DEV_POOL_PATH = WORK_DIR / 'dev_pool_features.parquet'
EVAL_POOL_PATH = WORK_DIR / 'eval_pool_features.parquet'
DEV_PAIRS_PATH = WORK_DIR / 'dev_pairs.parquet'
EVAL_PAIRS_PREP_PATH = WORK_DIR / 'eval_pairs.parquet'
DEV_PAIR_HANDS_PATH = WORK_DIR / 'dev_pair_hands.parquet'
EVAL_PAIR_HANDS_PATH = WORK_DIR / 'eval_pair_hands.parquet'
ACTION_CONTEXT_PATH = WORK_DIR / 'action_context.parquet'
PLAYER_HAND_PATH = WORK_DIR / 'player_hand_features.parquet'
PLAYER_ACTION_BASELINE_PATH = WORK_DIR / 'player_action_baselines.parquet'
DEV_HAND_FEATURES_PATH = WORK_DIR / 'dev_hand_features.parquet'
EVAL_HAND_FEATURES_PATH = WORK_DIR / 'eval_hand_features.parquet'
H2_DEV_SEMANTIC_HAND_PATH = WORK_DIR / 'h2_dev_semantic_hands.parquet'
H2_EVAL_SEMANTIC_HAND_PATH = WORK_DIR / 'h2_eval_semantic_hands.parquet'

print('DATA_DIR:', DATA_DIR)
print('WORK_DIR:', WORK_DIR)
print('Strict cache rebuild:', RESET_CACHE)
assert DATA_DIR.exists(), f'Missing competition data at {DATA_DIR}'


DATA_DIR: /kaggle/input/competitions/detect-suspicious-value-transfers-in-poker
WORK_DIR: /kaggle/working/poker_v3_h2_exact_semantics_20260908
Strict cache rebuild: True


In [2]:
# ============================================================
# 1. LOAD SOURCES + SCHEMA / ROW-COUNT REVISION
# ============================================================

players = pl.scan_parquet(PLAYERS_PATH)
hands = pl.scan_parquet(HANDS_PATH)
seats = pl.scan_parquet(SEATS_PATH)
actions = pl.scan_parquet(ACTIONS_PATH)

dev_labels = pl.read_csv(DEV_LABELS_PATH)
dev_evidence = pl.read_csv(DEV_EVIDENCE_PATH)
eval_pairs = pl.read_csv(EVAL_PAIRS_PATH)
sample_sub = pl.read_csv(SAMPLE_SUB_PATH)

required = {
    'players': {'player_id','account_age_days','experience_hands_bucket','preferred_stake','region_bucket','client_family'},
    'hands': {'hand_id','table_id','started_at','phase','button_seat','small_blind','big_blind','board_cards','final_pot','players_dealt','players_at_showdown'},
    'seats': {'hand_id','player_id','seat_no','starting_stack','hole_card_1','hole_card_2','total_contribution','net_chips','folded','went_to_showdown','won_share'},
    'actions': {'hand_id','action_no','street','player_id','action','amount','amount_to','pot_before','stack_before','to_call','players_active'},
    'dev_labels': {'pair_id','player_1','player_2','label','label_status','behavior_family'},
    'dev_evidence': {'pair_id','evidence_rank','hand_id','behavior_family'},
    'eval_pairs': {'pair_id','player_1','player_2','shared_hands'},
    'sample_sub': {'pair_id','risk_score','predicted_behavior','evidence_hand_1','evidence_hand_2','evidence_hand_3','evidence_hand_4','evidence_hand_5'},
}

actual = {
    'players': set(players.collect_schema().names()),
    'hands': set(hands.collect_schema().names()),
    'seats': set(seats.collect_schema().names()),
    'actions': set(actions.collect_schema().names()),
    'dev_labels': set(dev_labels.columns),
    'dev_evidence': set(dev_evidence.columns),
    'eval_pairs': set(eval_pairs.columns),
    'sample_sub': set(sample_sub.columns),
}

for name in required:
    missing = required[name] - actual[name]
    assert not missing, f'{name}: missing columns {sorted(missing)}'

counts = {
    'players': players.select(pl.len()).collect().item(),
    'hands': hands.select(pl.len()).collect().item(),
    'seats': seats.select(pl.len()).collect().item(),
    'actions': actions.select(pl.len()).collect().item(),
    'dev_labels': len(dev_labels),
    'dev_evidence': len(dev_evidence),
    'eval_pairs': len(eval_pairs),
    'sample_sub': len(sample_sub),
}

expected_counts = {
    'players': 12_000,
    'hands': 2_000_000,
    'seats': 12_000_000,
    'actions': 18_609_028,
    'dev_labels': 1_860,
    'dev_evidence': 1_817,
    'eval_pairs': 112_540,
    'sample_sub': 112_540,
}

for name, n in counts.items():
    mark = '✅' if n == expected_counts[name] else '⚠️'
    print(f'{mark} {name:<14} {n:>12,} rows')

n_pos = dev_labels.filter(pl.col('label') == 1).height
n_neg = dev_labels.filter(pl.col('label') == 0).height
print(f'Confirmed positives: {n_pos:,} | confirmed non-targets: {n_neg:,}')
assert n_pos == 372 and n_neg == 1488, 'Development label counts differ from the reviewed source.'
assert dev_evidence['pair_id'].n_unique() == 372, 'Evidence must cover the disclosed positive pairs.'
source_schema_checked = True
print('✅ Source/schema revision passed.')


✅ players              12,000 rows
✅ hands             2,000,000 rows
✅ seats            12,000,000 rows
✅ actions          18,609,028 rows
✅ dev_labels            1,860 rows
✅ dev_evidence          1,817 rows
✅ eval_pairs          112,540 rows
✅ sample_sub          112,540 rows
Confirmed positives: 372 | confirmed non-targets: 1,488
✅ Source/schema revision passed.


## 2. Exact competition metric

Copied in logic from the supplied metric notebook:

- 70% pair Average Precision
- 20% evidence MAP@5
- 10% behavior macro Average Precision over the three disclosed target families

Evidence truth is treated as a **set**, not as graded relevance by `evidence_rank`.

In [3]:
# ============================================================
# 2. EXACT METRIC HELPERS
# ============================================================

ALLOWED_BEHAVIORS = {
    'none', 'directed_transfer', 'soft_play',
    'coordinated_isolation', 'other_coordination'
}
TARGET_BEHAVIORS = ('directed_transfer','soft_play','coordinated_isolation')
PAIR_METRIC_WEIGHT = 0.70
EVIDENCE_METRIC_WEIGHT = 0.20
BEHAVIOR_METRIC_WEIGHT = 0.10
EVIDENCE_COLUMNS = tuple(f'evidence_hand_{i}' for i in range(1, 6))
NO_EVIDENCE = 'NO_EVIDENCE'


def _average_precision_exact(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    positives = int(y_true.sum())
    if positives == 0:
        return 0.0
    order = np.argsort(-scores, kind='mergesort')
    ranked = y_true[order]
    tp = np.cumsum(ranked)
    ranks = np.arange(1, len(ranked) + 1)
    return float(np.sum((tp / ranks) * ranked) / positives)


def _clean_evidence(values):
    out = []
    for v in values:
        if pd.isna(v):
            continue
        text = str(v).strip()
        if text and text != NO_EVIDENCE:
            out.append(text)
    return out


def competition_components(solution, submission):
    truth = solution.set_index('pair_id').sort_index()
    pred = submission.set_index('pair_id').loc[truth.index]

    risk = pred['risk_score'].astype(float).to_numpy()
    y = truth['risk_score'].astype(int).to_numpy()
    pair_ap = _average_precision_exact(y, risk)

    true_behavior = truth['predicted_behavior'].astype(str).to_numpy()
    pred_behavior = pred['predicted_behavior'].astype(str).to_numpy()
    behavior_scores = []
    for behavior in TARGET_BEHAVIORS:
        behavior_truth = (true_behavior == behavior).astype(int)
        if behavior_truth.sum() == 0:
            behavior_scores.append(0.0)
            continue
        behavior_risk = np.where(pred_behavior == behavior, risk, 0.0)
        behavior_scores.append(_average_precision_exact(behavior_truth, behavior_risk))
    behavior_map = float(np.mean(behavior_scores))

    ev_scores = []
    for pos in np.flatnonzero(y == 1):
        relevant = set(_clean_evidence(truth.iloc[pos][list(EVIDENCE_COLUMNS)].tolist()))
        submitted = _clean_evidence(pred.iloc[pos][list(EVIDENCE_COLUMNS)].tolist())
        if not relevant:
            ev_scores.append(0.0)
            continue
        hits = 0
        psum = 0.0
        for rank, hand_id in enumerate(submitted[:5], start=1):
            if hand_id in relevant:
                hits += 1
                psum += hits / rank
        ev_scores.append(psum / min(len(relevant), 5))
    evidence_map = float(np.mean(ev_scores)) if ev_scores else 0.0

    final = PAIR_METRIC_WEIGHT * pair_ap + EVIDENCE_METRIC_WEIGHT * evidence_map + BEHAVIOR_METRIC_WEIGHT * behavior_map
    return {
        'pair_ap': pair_ap,
        'evidence_map5': evidence_map,
        'behavior_map': behavior_map,
        'final_score': final,
        'behavior_class_ap': dict(zip(TARGET_BEHAVIORS, behavior_scores)),
    }


def evidence_map5(pair_ids, hand_ids, target, scores):
    """Within-pair MAP@5 with deterministic score-desc / hand_id-asc ties."""
    frame = pd.DataFrame({
        'pair_id': np.asarray(pair_ids, dtype=object),
        'hand_id': np.asarray(hand_ids, dtype=object),
        'target': np.asarray(target, dtype=np.int8),
        'score': np.asarray(scores, dtype=float),
    })
    vals = []
    for _, g in frame.groupby('pair_id', sort=False):
        g = g.sort_values(['score','hand_id'], ascending=[False,True], kind='mergesort')
        rel = g['target'].to_numpy()[:5]
        denom = min(int(g['target'].sum()), 5)
        vals.append(0.0 if denom == 0 else float(((np.cumsum(rel) / np.arange(1, len(rel)+1)) * rel).sum() / denom))
    return float(np.mean(vals)) if vals else 0.0

print('✅ Exact metric helpers ready.')


✅ Exact metric helpers ready.


## 3. Lightweight all-pair pool features

The EDA showed that **within-pool relative abnormality** is valuable, especially for directed transfer. We compute these features over *all* co-occurring pairs in each phase before selecting labeled/PU/evaluation pairs.

In [4]:
# ============================================================
# 3. ALL-PAIR POOL FEATURES (DUCKDB, MEMORY-SAFE)
# ============================================================


def build_pool_features(phase, out_path):
    if out_path.exists():
        return

    con = duckdb.connect()
    sql = f"""
    COPY (
        WITH phase_seats AS (
            SELECT
                s.hand_id,
                s.player_id,
                s.total_contribution,
                s.net_chips,
                h.table_id,
                h.big_blind,
                h.final_pot
            FROM read_parquet('{SEATS_PATH}') s
            JOIN read_parquet('{HANDS_PATH}') h USING(hand_id)
            WHERE h.phase = '{phase}'
        ),
        pair_hands AS (
            SELECT
                a.table_id,
                a.hand_id,
                a.player_id AS p1,
                b.player_id AS p2,
                a.net_chips * 1.0 / NULLIF(a.big_blind, 0) AS n1,
                b.net_chips * 1.0 / NULLIF(a.big_blind, 0) AS n2,
                a.total_contribution * 1.0 / NULLIF(a.big_blind, 0) AS c1,
                b.total_contribution * 1.0 / NULLIF(a.big_blind, 0) AS c2,
                a.final_pot * 1.0 / NULLIF(a.big_blind, 0) AS pot_bb
            FROM phase_seats a
            JOIN phase_seats b
              ON a.hand_id = b.hand_id
             AND a.player_id < b.player_id
        ),
        agg AS (
            SELECT
                table_id, p1, p2,
                COUNT(*) AS pool_shared_hands,
                AVG(ABS(n1-n2)) AS pool_mean_abs_net_gap_bb,
                MAX(ABS(n1-n2)) AS pool_max_abs_net_gap_bb,
                AVG(c1+c2) AS pool_mean_combined_contrib_bb,
                AVG(CASE WHEN pot_bb >= 20 THEN 1.0 ELSE 0.0 END) AS pool_big_pot_rate,
                SUM(n1-n2) AS pool_signed_net_diff_sum,
                SUM(ABS(n1-n2)) AS pool_abs_net_diff_sum,
                SUM(LEAST(GREATEST(-n1,0), GREATEST(n2,0))) AS pool_transfer_1_to_2,
                SUM(LEAST(GREATEST(-n2,0), GREATEST(n1,0))) AS pool_transfer_2_to_1
            FROM pair_hands
            GROUP BY table_id,p1,p2
        ),
        scored AS (
            SELECT
                *,
                ABS(pool_signed_net_diff_sum) / (pool_abs_net_diff_sum + 1e-6) AS pool_direction_consistency,
                ABS(pool_transfer_1_to_2-pool_transfer_2_to_1) /
                    (pool_transfer_1_to_2+pool_transfer_2_to_1+1e-6) AS pool_transfer_asymmetry
            FROM agg
        )
        SELECT
            *,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_shared_hands) AS pool_pct_shared_hands,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_mean_abs_net_gap_bb) AS pool_pct_net_gap,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_max_abs_net_gap_bb) AS pool_pct_max_net_gap,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_mean_combined_contrib_bb) AS pool_pct_combined_contrib,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_big_pot_rate) AS pool_pct_big_pot_rate,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_direction_consistency) AS pool_pct_direction_consistency,
            PERCENT_RANK() OVER (PARTITION BY table_id ORDER BY pool_transfer_asymmetry) AS pool_pct_transfer_asymmetry
        FROM scored
    ) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()

for phase, path in [('development', DEV_POOL_PATH), ('evaluation', EVAL_POOL_PATH)]:
    with tqdm(total=1, desc=f'Pool features: {phase}') as pbar:
        build_pool_features(phase, path)
        pbar.update()
    n = pl.scan_parquet(path).select(pl.len()).collect().item()
    print(f'{phase}: {n:,} all-pair rows')

print('✅ Pool-relative features ready.')

Pool features: development:   0%|          | 0/1 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

development: 173,862 all-pair rows


Pool features: evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

evaluation: 172,878 all-pair rows
✅ Pool-relative features ready.


## 4. Pair preparation and label-independent PU sampling

The review correctly identified that V2a selected a special unknown stratum using **all disclosed positive players before folds were assigned**. That makes CV preprocessing label-informed.

Strict V2.1 instead samples unknown pairs **without using labels**. During each training fold, unknown rows that touch a positive player are assigned the lower PU weight only when that player is positive in that fold's **training labels**. The validation labels never determine the training sample or training weight.


In [5]:
# ============================================================
# 4. PREPARE DEVELOPMENT / EVALUATION PAIRS + PAIR-HAND MAPS
#    STRICT: UNKNOWN SAMPLING IS LABEL-INDEPENDENT
# ============================================================

BEHAVIOR_TO_ID = {'none':0, 'directed_transfer':1, 'soft_play':2, 'coordinated_isolation':3}


def add_pair_key(df):
    return df.with_columns(
        pl.min_horizontal('player_1','player_2').alias('p_low'),
        pl.max_horizontal('player_1','player_2').alias('p_high')
    )


def all_pair_hands(phase):
    hand_players = (
        seats.join(
            hands.filter(pl.col('phase') == phase).select(['hand_id','table_id','started_at']),
            on='hand_id'
        )
        .group_by(['hand_id','table_id','started_at'])
        .agg(pl.col('player_id').sort_by('seat_no').alias('players'))
        .collect(engine='streaming')
        .sort(['table_id','started_at','hand_id'])
        .with_columns(
            (pl.col('started_at').rank('ordinal').over('table_id') /
             pl.len().over('table_id')).cast(pl.Float32).alias('phase_progress')
        )
    )

    frames = []
    for i, j in combinations(range(6), 2):
        frames.append(
            hand_players.lazy().select(
                ['hand_id','table_id','phase_progress',
                 pl.col('players').list.get(i).alias('a'),
                 pl.col('players').list.get(j).alias('b')]
            )
        )

    return (
        pl.concat(frames)
        .with_columns(
            pl.min_horizontal('a','b').alias('p_low'),
            pl.max_horizontal('a','b').alias('p_high')
        )
        .drop(['a','b'])
    )


dev_label_keys = add_pair_key(dev_labels)
eval_pair_keys = add_pair_key(eval_pairs)

required_cache = [DEV_PAIRS_PATH,EVAL_PAIRS_PREP_PATH,DEV_PAIR_HANDS_PATH,EVAL_PAIR_HANDS_PATH]

if not all(p.exists() for p in required_cache):
    dev_pool = pl.read_parquet(DEV_POOL_PATH).rename({'p1':'p_low','p2':'p_high'})

    dev_counts = dev_pool.select([
        'table_id','p_low','p_high',
        pl.col('pool_shared_hands').cast(pl.Int32).alias('shared_hands')
    ]).join(
        dev_label_keys.select(['pair_id','p_low','p_high','label','label_status','behavior_family']),
        on=['p_low','p_high'], how='left'
    )

    dev_min_shared = int(np.ceil(eval_pairs['shared_hands'].min() * 1.5))
    print('Development PU minimum shared hands:', dev_min_shared)

    known_pairs = (
        dev_counts.filter(pl.col('pair_id').is_not_null())
        .with_columns(
            pl.lit(True).alias('is_labeled'),
            pl.lit(False).alias('is_pu'),
            pl.lit('known').alias('pu_kind'),
            pl.lit(1.0, dtype=pl.Float32).alias('population_weight')
        )
    )

    unlabeled = dev_counts.filter(
        pl.col('pair_id').is_null() & (pl.col('shared_hands') >= dev_min_shared)
    )

    # This sample is selected BEFORE and WITHOUT consulting positive-player labels.
    unknown_sample = (
        unlabeled
        .sample(fraction=1, shuffle=True, seed=SEED+2)
        .group_by('table_id', maintain_order=True)
        .head(UNKNOWN_PER_TABLE)
    )

    population_counts = unlabeled.group_by('table_id').len().rename({'len':'_unknown_population_n'})
    sample_counts = unknown_sample.group_by('table_id').len().rename({'len':'_unknown_sample_n'})

    unknown_sample = (
        unknown_sample
        .join(population_counts,on='table_id',how='left')
        .join(sample_counts,on='table_id',how='left')
        .with_columns(
            pl.concat_str([pl.lit('U'), 'p_low', 'p_high'], separator='_').alias('pair_id'),
            pl.lit(False).alias('is_labeled'),
            pl.lit(True).alias('is_pu'),
            pl.lit('sampled_unknown').alias('pu_kind'),
            (pl.col('_unknown_population_n') / pl.col('_unknown_sample_n')).cast(pl.Float32).alias('population_weight')
        )
        .drop(['_unknown_population_n','_unknown_sample_n'])
    )

    dev_pairs_prep = (
        pl.concat([known_pairs, unknown_sample], how='diagonal_relaxed')
        .with_columns(pl.col('behavior_family').fill_null('unknown'))
        .with_columns(
            pl.col('behavior_family').replace(BEHAVIOR_TO_ID, default=-1).cast(pl.Int8).alias('behavior_id')
        )
        .rename({'p_low':'player_1','p_high':'player_2'})
        .sort(['table_id','pair_id'])
    )
    dev_pairs_prep.write_parquet(DEV_PAIRS_PATH)

    with tqdm(total=2, desc='Building selected pair-hand maps') as pbar:
        dev_all = all_pair_hands('development')
        dev_pair_hands = (
            dev_all.join(
                dev_pairs_prep.lazy().select([
                    'pair_id',
                    pl.col('player_1').alias('p_low'),
                    pl.col('player_2').alias('p_high')
                ]),
                on=['p_low','p_high'], how='inner'
            )
            .select(['pair_id','hand_id','table_id','phase_progress'])
            .collect(engine='streaming')
        )
        dev_pair_hands.write_parquet(DEV_PAIR_HANDS_PATH)
        del dev_all, dev_pair_hands
        gc.collect()
        pbar.update()

        eval_all = all_pair_hands('evaluation')
        eval_pair_hands = (
            eval_all.join(
                eval_pair_keys.lazy().select(['pair_id','p_low','p_high']),
                on=['p_low','p_high'], how='inner'
            )
            .select(['pair_id','hand_id','table_id','phase_progress'])
            .collect(engine='streaming')
        )
        eval_tables = eval_pair_hands.group_by('pair_id').agg(
            pl.col('table_id').first(),
            pl.len().cast(pl.Int32).alias('shared_hands_calc')
        )
        eval_pairs_prep = (
            eval_pair_keys.select([
                'pair_id',
                pl.col('p_low').alias('player_1'),
                pl.col('p_high').alias('player_2'),
                'shared_hands'
            ])
            .join(eval_tables, on='pair_id', how='left')
            .with_columns(
                pl.lit(False).alias('is_labeled'),
                pl.lit(False).alias('is_pu'),
                pl.lit('evaluation').alias('pu_kind'),
                pl.lit(1.0, dtype=pl.Float32).alias('population_weight'),
                pl.lit(None, dtype=pl.Int8).alias('label'),
                pl.lit('unknown').alias('behavior_family'),
                pl.lit(-1, dtype=pl.Int8).alias('behavior_id')
            )
            .sort(['table_id','pair_id'])
        )
        eval_pair_hands.write_parquet(EVAL_PAIR_HANDS_PATH)
        eval_pairs_prep.write_parquet(EVAL_PAIRS_PREP_PATH)
        del eval_all, eval_pair_hands
        gc.collect()
        pbar.update()


dev_pairs_prep = pl.read_parquet(DEV_PAIRS_PATH)
eval_pairs_prep = pl.read_parquet(EVAL_PAIRS_PREP_PATH)

print(dev_pairs_prep.group_by('pu_kind').len().sort('len', descending=True))
print(f'Development pairs: {len(dev_pairs_prep):,}')
print(f'Evaluation pairs:  {len(eval_pairs_prep):,}')

# Fail early on missing reconstruction rows; null != value can otherwise evade a mismatch filter.
assert eval_pairs_prep['table_id'].null_count() == 0, 'Evaluation pair reconstruction has null table_id.'
assert eval_pairs_prep['shared_hands_calc'].null_count() == 0, 'Evaluation pair reconstruction has null shared_hands_calc.'
assert dev_pairs_prep['population_weight'].null_count() == 0, 'Development population weights contain nulls.'
assert (dev_pairs_prep['population_weight'] > 0).all(), 'Development population weights must be positive.'

mismatch = eval_pairs_prep.filter(pl.col('shared_hands') != pl.col('shared_hands_calc')).height
assert mismatch == 0, f'Evaluation shared-hand reconstruction mismatch: {mismatch}'

# Development reconstruction must cover every disclosed label pair and every evidence key.
dev_pair_hand_keys = pl.read_parquet(DEV_PAIR_HANDS_PATH).select(['pair_id','hand_id']).unique()
dev_reconstructed_pairs = dev_pair_hand_keys.select('pair_id').unique()
missing_dev_label_pairs = (
    dev_labels.select('pair_id').unique()
    .join(dev_reconstructed_pairs,on='pair_id',how='anti')
)
assert missing_dev_label_pairs.height == 0, (
    f'{missing_dev_label_pairs.height} development label pairs have no reconstructed shared hand.'
)

dev_evidence_keys_unique = dev_evidence.select(['pair_id','hand_id']).unique()
unmatched_dev_evidence = dev_evidence_keys_unique.join(
    dev_pair_hand_keys,on=['pair_id','hand_id'],how='anti'
)
assert unmatched_dev_evidence.height == 0, (
    f'{unmatched_dev_evidence.height} development evidence keys are absent from the reconstructed pair-hand map.'
)
assert dev_evidence_keys_unique.height == len(dev_evidence), 'Duplicate development evidence (pair_id, hand_id) keys detected.'
development_evidence_membership_validated = True

print('✅ Pair preparation passed with label-independent PU sampling.')
print('✅ Development label/evidence membership reconstruction passed.')


Development PU minimum shared hands: 57


Building selected pair-hand maps:   0%|          | 0/2 [00:00<?, ?it/s]

shape: (2, 2)
┌─────────────────┬───────┐
│ pu_kind         ┆ len   │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ sampled_unknown ┆ 28800 │
│ known           ┆ 1860  │
└─────────────────┴───────┘
Development pairs: 30,660
Evaluation pairs:  112,540
✅ Pair preparation passed with label-independent PU sampling.
✅ Development label/evidence membership reconstruction passed.


## 5. Ordered action context and player-hand features

This is the main forensic layer. V2 preserves V1 features and adds:

- river checks / river aggression;
- overbets;
- call-price and bet-to-pot accumulators;
- previous/next actors and actions for exact sequence motifs;
- richer hole-card proxies (pair, suited, rank sum, gap, broadway, premium pair).

In [6]:
# ============================================================
# 5A. ORDERED ACTION CONTEXT
# ============================================================

AGGRESSIVE_ACTIONS = ['bet','raise']

if not ACTION_CONTEXT_PATH.exists():
    with tqdm(total=1, desc='Ordered action context') as pbar:
        action_context = (
            actions.join(
                hands.select(['hand_id','phase','big_blind']),
                on='hand_id'
            )
            .sort(['hand_id','action_no'])
            .with_columns(
                (
                    pl.col('action').is_in(AGGRESSIVE_ACTIONS) |
                    ((pl.col('action') == 'all_in') & (pl.col('amount') > pl.col('to_call')))
                ).alias('is_aggressive'),
                (pl.col('amount') / pl.col('big_blind')).cast(pl.Float32).alias('amount_bb'),
                (pl.col('to_call') / pl.col('big_blind')).cast(pl.Float32).alias('to_call_bb'),
                (pl.col('amount') / pl.max_horizontal('pot_before','big_blind')).clip(0,20).cast(pl.Float32).alias('amount_pot_ratio'),
                (pl.col('to_call') / pl.max_horizontal(pl.col('pot_before') + pl.col('to_call'), pl.lit(1))).clip(0,1).cast(pl.Float32).alias('call_price'),
                (pl.col('stack_before') / pl.max_horizontal('pot_before','big_blind')).clip(0,500).cast(pl.Float32).alias('stack_to_pot')
            )
            .with_columns(
                pl.when(pl.col('is_aggressive')).then(pl.col('player_id')).otherwise(None).alias('_aggressor')
            )
            .with_columns(
                pl.col('_aggressor').shift(1).forward_fill().over(['hand_id','street']).alias('last_aggressor'),
                pl.col('player_id').shift(1).over('hand_id').alias('prev_player_id'),
                pl.col('action').shift(1).over('hand_id').alias('prev_action'),
                pl.col('player_id').shift(-1).over('hand_id').alias('next_player_id'),
                pl.col('action').shift(-1).over('hand_id').alias('next_action'),
                pl.col('players_active').shift(-1).over('hand_id').alias('next_players_active'),
                pl.col('players_active').shift(-2).over('hand_id').alias('next2_players_active')
            )
            .select([
                'hand_id','phase','action_no','street','player_id','action','to_call','players_active',
                'is_aggressive','last_aggressor','prev_player_id','prev_action','next_player_id','next_action',
                'next_players_active','next2_players_active','amount_bb','to_call_bb','amount_pot_ratio','call_price','stack_to_pot'
            ])
        )
        action_context.sink_parquet(ACTION_CONTEXT_PATH)
        pbar.update()

print('Action context rows:', pl.scan_parquet(ACTION_CONTEXT_PATH).select(pl.len()).collect().item())

Ordered action context:   0%|          | 0/1 [00:00<?, ?it/s]

Action context rows: 18609028


In [7]:
# ============================================================
# 5B. PLAYER-HAND FEATURES + ACTION BASELINES
# ============================================================


def card_rank_expr(column):
    rank = pl.col(column).cast(pl.Utf8).str.replace(r'[cdhsCDHS]$', '').str.to_uppercase()
    return (
        pl.when(rank == 'A').then(14)
        .when(rank == 'K').then(13)
        .when(rank == 'Q').then(12)
        .when(rank == 'J').then(11)
        .when(rank == 'T').then(10)
        .otherwise(rank.cast(pl.Int8, strict=False))
    )

ACTION_COUNT_COLS = [
    'n_actions','aggressive_actions','bets','raises','calls','checks','folds','all_ins',
    'postflop_checks','postflop_calls','river_checks','river_aggression','preflop_raises',
    'headsup_actions_raw','overbets','aggressive_size_count','call_price_count'
]
ACTION_SUM_COLS = ['aggressive_size_sum','call_price_sum']
ACTION_MAX_COLS = ['max_amount_bb','max_to_call_bb','max_amount_pot_ratio']
ACTION_FEATURES = ACTION_COUNT_COLS + ACTION_SUM_COLS + ACTION_MAX_COLS

if not PLAYER_HAND_PATH.exists():
    with tqdm(total=1, desc='Player-hand features') as pbar:
        ac = pl.scan_parquet(ACTION_CONTEXT_PATH)
        action_by_player = ac.group_by(['hand_id','player_id']).agg(
            pl.len().cast(pl.Int16).alias('n_actions'),
            pl.col('is_aggressive').sum().cast(pl.Int16).alias('aggressive_actions'),
            (pl.col('action') == 'bet').sum().cast(pl.Int16).alias('bets'),
            (pl.col('action') == 'raise').sum().cast(pl.Int16).alias('raises'),
            (pl.col('action') == 'call').sum().cast(pl.Int16).alias('calls'),
            (pl.col('action') == 'check').sum().cast(pl.Int16).alias('checks'),
            (pl.col('action') == 'fold').sum().cast(pl.Int16).alias('folds'),
            (pl.col('action') == 'all_in').sum().cast(pl.Int16).alias('all_ins'),
            ((pl.col('street') != 'preflop') & (pl.col('action') == 'check')).sum().cast(pl.Int16).alias('postflop_checks'),
            ((pl.col('street') != 'preflop') & (pl.col('action') == 'call')).sum().cast(pl.Int16).alias('postflop_calls'),
            ((pl.col('street') == 'river') & (pl.col('action') == 'check')).sum().cast(pl.Int16).alias('river_checks'),
            ((pl.col('street') == 'river') & pl.col('is_aggressive')).sum().cast(pl.Int16).alias('river_aggression'),
            ((pl.col('street') == 'preflop') & (pl.col('action') == 'raise')).sum().cast(pl.Int16).alias('preflop_raises'),
            (pl.col('players_active') == 2).sum().cast(pl.Int16).alias('headsup_actions_raw'),
            (pl.col('is_aggressive') & (pl.col('amount_pot_ratio') > 1)).sum().cast(pl.Int16).alias('overbets'),
            pl.when(pl.col('is_aggressive')).then(pl.col('amount_pot_ratio')).otherwise(0).sum().cast(pl.Float32).alias('aggressive_size_sum'),
            pl.col('is_aggressive').sum().cast(pl.Int16).alias('aggressive_size_count'),
            pl.when(pl.col('action') == 'call').then(pl.col('call_price')).otherwise(0).sum().cast(pl.Float32).alias('call_price_sum'),
            (pl.col('action') == 'call').sum().cast(pl.Int16).alias('call_price_count'),
            pl.col('amount_bb').max().cast(pl.Float32).alias('max_amount_bb'),
            pl.col('to_call_bb').max().cast(pl.Float32).alias('max_to_call_bb'),
            pl.col('amount_pot_ratio').max().cast(pl.Float32).alias('max_amount_pot_ratio')
        )

        player_hands = (
            seats.join(
                hands.select(['hand_id','phase','big_blind','final_pot']),
                on='hand_id'
            )
            .with_columns(
                card_rank_expr('hole_card_1').alias('r1'),
                card_rank_expr('hole_card_2').alias('r2'),
                (pl.col('starting_stack') / pl.col('big_blind')).cast(pl.Float32).alias('stack_bb'),
                (pl.col('total_contribution') / pl.col('big_blind')).cast(pl.Float32).alias('contribution_bb'),
                (pl.col('net_chips') / pl.col('big_blind')).cast(pl.Float32).alias('net_bb'),
                (pl.col('final_pot') / pl.col('big_blind')).cast(pl.Float32).alias('pot_bb')
            )
            .with_columns(
                (pl.col('r1') == pl.col('r2')).cast(pl.Int8).alias('hole_pair'),
                (pl.col('hole_card_1').str.slice(-1,1) == pl.col('hole_card_2').str.slice(-1,1)).cast(pl.Int8).alias('hole_suited'),
                (pl.col('r1') + pl.col('r2')).cast(pl.Int16).alias('hole_rank_sum'),
                (pl.col('r1') - pl.col('r2')).abs().cast(pl.Int8).alias('hole_rank_gap'),
                ((pl.col('r1') >= 10).cast(pl.Int8) + (pl.col('r2') >= 10).cast(pl.Int8)).alias('broadway_count'),
                ((pl.col('r1') == pl.col('r2')) & (pl.min_horizontal('r1','r2') >= 10)).cast(pl.Int8).alias('premium_pair')
            )
            .with_columns(
                (
                    pl.max_horizontal('r1','r2')
                    + 5.0 * pl.col('hole_pair')
                    + 1.5 * pl.col('hole_suited')
                    + 0.20 * pl.col('hole_rank_sum')
                    - 0.15 * pl.col('hole_rank_gap')
                    + 1.0 * (pl.col('broadway_count') == 2).cast(pl.Int8)
                ).cast(pl.Float32).alias('hole_strength_v2')
            )
            .join(action_by_player, on=['hand_id','player_id'], how='left')
            .with_columns([pl.col(c).fill_null(0) for c in ACTION_FEATURES])
            .select([
                'hand_id','player_id','phase','pot_bb','stack_bb','contribution_bb','net_bb',
                'folded','went_to_showdown','won_share',
                'hole_pair','hole_suited','hole_rank_sum','hole_rank_gap','broadway_count','premium_pair','hole_strength_v2',
                *ACTION_FEATURES
            ])
        )
        player_hands.sink_parquet(PLAYER_HAND_PATH)
        pbar.update()

if not PLAYER_ACTION_BASELINE_PATH.exists():
    with tqdm(total=1, desc='Player action baselines') as pbar:
        ac = pl.scan_parquet(ACTION_CONTEXT_PATH)
        baselines = ac.group_by(['phase','player_id']).agg(
            pl.len().alias('base_n_actions'),
            (pl.col('action') == 'fold').mean().cast(pl.Float32).alias('base_fold_rate'),
            (pl.col('action') == 'check').mean().cast(pl.Float32).alias('base_check_rate'),
            (pl.col('action') == 'call').mean().cast(pl.Float32).alias('base_call_rate'),
            (pl.col('action') == 'bet').mean().cast(pl.Float32).alias('base_bet_rate'),
            (pl.col('action') == 'raise').mean().cast(pl.Float32).alias('base_raise_rate'),
            pl.col('is_aggressive').mean().cast(pl.Float32).alias('base_aggression_rate'),
            (pl.col('players_active') == 2).mean().cast(pl.Float32).alias('base_headsup_rate'),
            pl.col('amount_pot_ratio').filter(pl.col('is_aggressive')).mean().cast(pl.Float32).alias('base_bet_to_pot'),
            pl.col('call_price').filter(pl.col('action') == 'call').mean().cast(pl.Float32).alias('base_call_price')
        )
        baselines.sink_parquet(PLAYER_ACTION_BASELINE_PATH)
        pbar.update()

print('Player-hand rows:', pl.scan_parquet(PLAYER_HAND_PATH).select(pl.len()).collect().item())
print('Baseline rows:', pl.scan_parquet(PLAYER_ACTION_BASELINE_PATH).select(pl.len()).collect().item())
print('✅ Player forensic layer ready.')

Player-hand features:   0%|          | 0/1 [00:00<?, ?it/s]

Player action baselines:   0%|          | 0/1 [00:00<?, ?it/s]

Player-hand rows: 12000000
Baseline rows: 24000
✅ Player forensic layer ready.


## 6. Advanced pair-hand features

Key new features are intentionally named so the source EDA can verify them directly:

- `opposite_net_sign`
- `pair_river_checks`
- `immediate_outsider_folds_after_aggression`
- `directed_score_v2`, `soft_score_v2`, `isolation_score_v2`

In [8]:
# ============================================================
# 6. BUILD ADVANCED PAIR-HAND FEATURES
# ============================================================

PLAYER_COLS = [
    'pot_bb','stack_bb','contribution_bb','net_bb','folded','went_to_showdown','won_share',
    'hole_pair','hole_suited','hole_rank_sum','hole_rank_gap','broadway_count','premium_pair','hole_strength_v2',
    *ACTION_FEATURES
]


def player_side(side, player_col):
    return pl.scan_parquet(PLAYER_HAND_PATH).select(
        'hand_id',
        pl.col('player_id').alias(player_col),
        *[pl.col(c).alias(f'{c}_{side}') for c in PLAYER_COLS]
    )


def build_pair_hand_features(pair_hands_path, pairs_path, out_path):
    if out_path.exists():
        return

    pair_hands = pl.scan_parquet(pair_hands_path)
    pairs_lf = pl.scan_parquet(pairs_path)
    pair_base = pair_hands.join(
        pairs_lf.select(['pair_id','player_1','player_2']), on='pair_id'
    )

    p1 = player_side('1','player_1')
    p2 = player_side('2','player_2')

    hand = (
        pair_base.join(p1, on=['hand_id','player_1'])
        .join(p2, on=['hand_id','player_2'])
        .with_columns(
            (pl.col('contribution_bb_1') + pl.col('contribution_bb_2')).alias('pair_contribution_bb'),
            (pl.col('contribution_bb_1') - pl.col('contribution_bb_2')).abs().alias('contribution_gap_bb'),
            (pl.col('net_bb_1') - pl.col('net_bb_2')).alias('signed_net_diff_bb'),
            (pl.col('net_bb_1') - pl.col('net_bb_2')).abs().alias('net_gap_bb'),
            pl.max_horizontal('net_bb_1','net_bb_2').alias('max_win_bb'),
            (-pl.min_horizontal('net_bb_1','net_bb_2')).alias('max_loss_bb'),
            pl.min_horizontal((-pl.col('net_bb_1')).clip(lower_bound=0), pl.col('net_bb_2').clip(lower_bound=0)).alias('transfer_1_to_2_bb'),
            pl.min_horizontal((-pl.col('net_bb_2')).clip(lower_bound=0), pl.col('net_bb_1').clip(lower_bound=0)).alias('transfer_2_to_1_bb'),
            (((pl.col('net_bb_1') > 0) & (pl.col('net_bb_2') < 0)) |
             ((pl.col('net_bb_2') > 0) & (pl.col('net_bb_1') < 0))).cast(pl.Int8).alias('opposite_net_sign'),
            (pl.col('went_to_showdown_1') & pl.col('went_to_showdown_2')).cast(pl.Int8).alias('both_showdown'),
            (pl.col('folded_1') ^ pl.col('folded_2')).cast(pl.Int8).alias('one_folded'),
            pl.max_horizontal('hole_strength_v2_1','hole_strength_v2_2').alias('max_hole_strength_v2'),
            ((pl.col('hole_rank_sum_1') + pl.col('hole_rank_sum_2')) / 2).cast(pl.Float32).alias('mean_hole_rank_sum'),
            pl.max_horizontal('hole_pair_1','hole_pair_2').alias('any_pocket_pair'),
            ((pl.col('hole_pair_1') == 1) & (pl.col('hole_pair_2') == 1)).cast(pl.Int8).alias('both_pocket_pair'),
            pl.max_horizontal('hole_suited_1','hole_suited_2').alias('any_suited'),
            (pl.col('aggressive_actions_1') + pl.col('aggressive_actions_2')).alias('pair_aggression'),
            (pl.col('bets_1') + pl.col('bets_2')).alias('pair_bets'),
            (pl.col('raises_1') + pl.col('raises_2')).alias('pair_raises'),
            (pl.col('calls_1') + pl.col('calls_2')).alias('pair_calls'),
            (pl.col('checks_1') + pl.col('checks_2')).alias('pair_checks'),
            (pl.col('postflop_checks_1') + pl.col('postflop_checks_2')).alias('pair_postflop_checks'),
            (pl.col('river_checks_1') + pl.col('river_checks_2')).alias('pair_river_checks'),
            (pl.col('river_aggression_1') + pl.col('river_aggression_2')).alias('pair_river_aggression'),
            (pl.col('overbets_1') + pl.col('overbets_2')).alias('pair_overbets'),
            pl.max_horizontal('max_amount_bb_1','max_amount_bb_2').alias('max_amount_bb'),
            pl.max_horizontal('max_to_call_bb_1','max_to_call_bb_2').alias('max_to_call_bb'),
            pl.max_horizontal('max_amount_pot_ratio_1','max_amount_pot_ratio_2').alias('max_amount_pot_ratio')
        )
    )

    members = pl.concat([
        pair_base.select([
            'pair_id','hand_id',
            pl.col('player_1').alias('member'),
            pl.col('player_2').alias('partner'),
            pl.lit(1).alias('is_p1')
        ]),
        pair_base.select([
            'pair_id','hand_id',
            pl.col('player_2').alias('member'),
            pl.col('player_1').alias('partner'),
            pl.lit(0).alias('is_p1')
        ])
    ])

    ac = pl.scan_parquet(ACTION_CONTEXT_PATH)
    fold_at = (
        ac.filter(pl.col('action') == 'fold')
        .group_by(['hand_id','player_id'])
        .agg(pl.col('action_no').min().alias('partner_fold_no'))
        .rename({'player_id':'partner'})
    )

    member_actions = (
        members.join(ac, left_on=['hand_id','member'], right_on=['hand_id','player_id'], how='left')
        .join(fold_at, on=['hand_id','partner'], how='left')
        .with_columns(
            (
                (pl.col('players_active') == 2) &
                (pl.col('partner_fold_no').is_null() | (pl.col('partner_fold_no') > pl.col('action_no')))
            ).alias('true_pair_hu'),
            (
                pl.col('is_aggressive') &
                (pl.col('next_action') == 'fold') &
                pl.col('next_player_id').is_not_null() &
                (pl.col('next_player_id') != pl.col('partner')) &
                (pl.col('next_player_id') != pl.col('member'))
            ).alias('immediate_outsider_fold'),
            (
                pl.col('is_aggressive') &
                (pl.col('next_action') == 'fold') &
                (pl.col('next_player_id') == pl.col('partner'))
            ).alias('immediate_partner_fold'),
            (
                (pl.col('prev_action') == 'fold') &
                pl.col('prev_player_id').is_not_null() &
                (pl.col('prev_player_id') != pl.col('partner')) &
                (pl.col('prev_player_id') != pl.col('member'))
            ).alias('outsider_fold_then_pair_action')
        )
    )

    interaction = member_actions.group_by(['pair_id','hand_id']).agg(
        pl.col('true_pair_hu').sum().alias('true_hu_actions'),
        (pl.col('true_pair_hu') & (pl.col('action') == 'check')).sum().alias('true_hu_checks'),
        (pl.col('true_pair_hu') & (pl.col('action') == 'call')).sum().alias('true_hu_calls'),
        (pl.col('true_pair_hu') & pl.col('is_aggressive')).sum().alias('true_hu_aggression'),
        (pl.col('true_pair_hu') & pl.col('action').is_in(['check','call']) & (pl.col('is_p1') == 1)).sum().alias('p1_hu_passive'),
        (pl.col('true_pair_hu') & pl.col('action').is_in(['check','call']) & (pl.col('is_p1') == 0)).sum().alias('p2_hu_passive'),
        pl.col('immediate_outsider_fold').sum().alias('immediate_outsider_folds_after_aggression'),
        (pl.col('immediate_outsider_fold') & (pl.col('street') == 'preflop')).sum().alias('preflop_outsider_folds_after_aggression'),
        (pl.col('immediate_outsider_fold') & (pl.col('street') != 'preflop')).sum().alias('postflop_outsider_folds_after_aggression'),
        pl.col('immediate_partner_fold').sum().alias('immediate_partner_folds_after_aggression'),
        pl.col('outsider_fold_then_pair_action').sum().alias('outsider_fold_then_pair_actions')
    )

    responses = ac.filter(
        (pl.col('to_call') > 0) & pl.col('last_aggressor').is_not_null()
    ).select(
        'hand_id',
        pl.col('last_aggressor').alias('member'),
        pl.col('player_id').alias('responder'),
        'action','is_aggressive'
    )

    pressure = (
        members.join(responses, on=['hand_id','member'], how='inner')
        .with_columns((pl.col('responder') == pl.col('partner')).alias('partner_response'))
        .group_by(['pair_id','hand_id']).agg(
            (pl.col('partner_response') & (pl.col('action') == 'fold')).sum().alias('partner_folds_to_pair'),
            (pl.col('partner_response') & pl.col('action').is_in(['call','all_in']) & ~pl.col('is_aggressive')).sum().alias('partner_calls_to_pair'),
            (pl.col('partner_response') & pl.col('is_aggressive')).sum().alias('partner_raises_to_pair'),
            (~pl.col('partner_response') & (pl.col('action') == 'fold')).sum().alias('outsider_folds_to_pair'),
            (~pl.col('partner_response') & (pl.col('action') == 'call')).sum().alias('outsider_calls_to_pair'),
            (~pl.col('partner_response') & pl.col('is_aggressive')).sum().alias('outsider_raises_to_pair')
        )
    )

    fill_cols = [
        'true_hu_actions','true_hu_checks','true_hu_calls','true_hu_aggression','p1_hu_passive','p2_hu_passive',
        'immediate_outsider_folds_after_aggression','preflop_outsider_folds_after_aggression','postflop_outsider_folds_after_aggression',
        'immediate_partner_folds_after_aggression','outsider_fold_then_pair_actions',
        'partner_folds_to_pair','partner_calls_to_pair','partner_raises_to_pair',
        'outsider_folds_to_pair','outsider_calls_to_pair','outsider_raises_to_pair'
    ]

    result = (
        hand.join(interaction, on=['pair_id','hand_id'], how='left')
        .join(pressure, on=['pair_id','hand_id'], how='left')
        .with_columns([pl.col(c).fill_null(0) for c in fill_cols])
        .with_columns(
            pl.max_horizontal('transfer_1_to_2_bb','transfer_2_to_1_bb').alias('transfer_any_bb'),
            (pl.col('pair_contribution_bb') / (pl.col('pot_bb_1') + 1e-3)).clip(0,2).alias('pair_pot_share')
        )
        .with_columns(
            (
                pl.col('net_gap_bb').log1p()
                + 0.65 * pl.col('pair_contribution_bb').log1p()
                + 1.20 * pl.col('opposite_net_sign')
                + 0.45 * pl.col('true_hu_actions').log1p()
                + 0.35 * (pl.col('pair_bets') + pl.col('pair_calls')).log1p()
                + 0.35 * pl.col('pair_river_aggression').log1p()
            ).alias('directed_score_v2'),
            (
                1.35 * pl.col('pair_river_checks').log1p()
                + 0.80 * (pl.col('true_hu_checks') + pl.col('true_hu_calls')).log1p()
                + 0.55 * pl.col('pair_calls').log1p()
                + 0.65 * pl.col('any_pocket_pair')
                + 0.08 * pl.col('mean_hole_rank_sum')
                + 0.25 * pl.col('both_showdown')
                - 0.10 * pl.col('max_amount_pot_ratio')
            ).alias('soft_score_v2'),
            (
                1.80 * pl.col('immediate_outsider_folds_after_aggression').log1p()
                + 0.75 * pl.col('pair_raises').log1p()
                + 0.45 * pl.col('pair_overbets').log1p()
                + 0.35 * pl.col('outsider_folds_to_pair').log1p()
                + 0.20 * pl.col('max_amount_bb').log1p()
            ).alias('isolation_score_v2')
        )
        .with_columns(
            pl.max_horizontal('directed_score_v2','soft_score_v2','isolation_score_v2').alias('suspicious_score_v2')
        )
    )

    # Keep raw member action summaries for counterparty-relative aggregation.
    member_keep = []
    for side in ('1','2'):
        for c in [
            'n_actions','aggressive_actions','bets','raises','calls','checks','folds','headsup_actions_raw',
            'aggressive_size_sum','aggressive_size_count','call_price_sum','call_price_count'
        ]:
            member_keep.append(f'{c}_{side}')

    keep = [
        'pair_id','hand_id','table_id','phase_progress','pot_bb_1',
        'pair_contribution_bb','pair_pot_share','contribution_gap_bb','signed_net_diff_bb','net_gap_bb',
        'max_win_bb','max_loss_bb','transfer_1_to_2_bb','transfer_2_to_1_bb','transfer_any_bb','opposite_net_sign',
        'both_showdown','one_folded','max_hole_strength_v2','mean_hole_rank_sum','any_pocket_pair','both_pocket_pair','any_suited',
        'pair_aggression','pair_bets','pair_raises','pair_calls','pair_checks','pair_postflop_checks','pair_river_checks','pair_river_aggression','pair_overbets',
        'max_amount_bb','max_to_call_bb','max_amount_pot_ratio',
        *fill_cols,
        'directed_score_v2','soft_score_v2','isolation_score_v2','suspicious_score_v2',
        *member_keep
    ]

    result.select(keep).rename({'pot_bb_1':'pot_bb'}).sink_parquet(out_path)

jobs = [
    (DEV_PAIR_HANDS_PATH,DEV_PAIRS_PATH,DEV_HAND_FEATURES_PATH,'Development'),
    (EVAL_PAIR_HANDS_PATH,EVAL_PAIRS_PREP_PATH,EVAL_HAND_FEATURES_PATH,'Evaluation')
]

for pair_hands_path, pairs_path, out_path, name in tqdm(jobs, desc='Advanced pair-hand features'):
    build_pair_hand_features(pair_hands_path,pairs_path,out_path)
    n = pl.scan_parquet(out_path).select(pl.len()).collect().item()
    print(f'{name}: {n:,} pair-hand rows')

print('✅ Advanced pair-hand tables ready.')

Advanced pair-hand features:   0%|          | 0/2 [00:00<?, ?it/s]

Development: 3,704,105 pair-hand rows
Evaluation: 9,651,820 pair-hand rows
✅ Advanced pair-hand tables ready.


## H2 early preflight

Checks the semantic source schema, pair/player mappings, Numba availability, and a small data smoke test **before** pair/evidence CV.


In [9]:
# ============================================================
# H2 EARLY PREFLIGHT — FAIL BEFORE EXPENSIVE CV/TRAINING
# ============================================================

H2_REQUIRED_HAND_SOURCE_COLS = [
    'pair_id','hand_id',
    'n_actions_1','n_actions_2','aggressive_actions_1','aggressive_actions_2',
    'checks_1','checks_2','calls_1','calls_2','folds_1','folds_2',
    'pair_aggression','pair_raises','pair_calls','pair_checks','pair_river_checks','pair_river_aggression',
    'true_hu_actions','true_hu_checks','true_hu_calls','true_hu_aggression',
    'p1_hu_passive','p2_hu_passive',
    'immediate_outsider_folds_after_aggression','postflop_outsider_folds_after_aggression',
    'outsider_folds_to_pair','pair_overbets',
    'transfer_1_to_2_bb','transfer_2_to_1_bb','transfer_any_bb',
    'net_gap_bb','pair_contribution_bb','opposite_net_sign','max_amount_bb'
]

_h2_dev_schema = set(pl.scan_parquet(DEV_HAND_FEATURES_PATH).collect_schema().names())
_h2_eval_schema = set(pl.scan_parquet(EVAL_HAND_FEATURES_PATH).collect_schema().names())
_h2_missing_dev = sorted(set(H2_REQUIRED_HAND_SOURCE_COLS) - _h2_dev_schema)
_h2_missing_eval = sorted(set(H2_REQUIRED_HAND_SOURCE_COLS) - _h2_eval_schema)

assert H2_NUMBA_PRECHECK, 'Numba unavailable.'
assert not _h2_missing_dev, f'H2 development hand source is missing: {_h2_missing_dev}'
assert not _h2_missing_eval, f'H2 evaluation hand source is missing: {_h2_missing_eval}'
assert {'pair_id','player_1','player_2'}.issubset(dev_pairs_prep.columns)
assert {'pair_id','player_1','player_2'}.issubset(eval_pairs_prep.columns)
assert dev_pairs_prep['pair_id'].n_unique() == len(dev_pairs_prep)
assert eval_pairs_prep['pair_id'].n_unique() == len(eval_pairs_prep)

# Tiny smoke test: verify a real labeled pair maps to two seats, cards, board, and baselines.
_h2_smoke_pair = dev_labels['pair_id'][0]
_h2_smoke_pair_row = dev_pairs_prep.filter(pl.col('pair_id') == _h2_smoke_pair)
assert len(_h2_smoke_pair_row) == 1
_h2_smoke_p1 = _h2_smoke_pair_row['player_1'][0]
_h2_smoke_p2 = _h2_smoke_pair_row['player_2'][0]
_h2_smoke_hands = (
    pl.scan_parquet(DEV_HAND_FEATURES_PATH)
    .filter(pl.col('pair_id') == _h2_smoke_pair)
    .select('hand_id').head(3).collect()
)
assert len(_h2_smoke_hands) > 0
_h2_smoke_ids = _h2_smoke_hands['hand_id'].to_list()

_h2_smoke_seats = (
    pl.scan_parquet(SEATS_PATH)
    .filter(
        pl.col('hand_id').is_in(_h2_smoke_ids)
        & pl.col('player_id').is_in([_h2_smoke_p1,_h2_smoke_p2])
    )
    .select(['hand_id','player_id','hole_card_1','hole_card_2'])
    .collect()
)
assert _h2_smoke_seats['player_id'].n_unique() == 2
assert _h2_smoke_seats['hole_card_1'].null_count() == 0
assert _h2_smoke_seats['hole_card_2'].null_count() == 0

_h2_smoke_board = (
    pl.scan_parquet(HANDS_PATH)
    .filter(pl.col('hand_id').is_in(_h2_smoke_ids))
    .select(['hand_id','board_cards']).collect()
)
assert len(_h2_smoke_board) == len(_h2_smoke_ids)

_h2_smoke_baselines = (
    pl.scan_parquet(PLAYER_ACTION_BASELINE_PATH)
    .filter(
        (pl.col('phase') == 'development')
        & pl.col('player_id').is_in([_h2_smoke_p1,_h2_smoke_p2])
    )
    .select('player_id').unique().collect()
)
assert _h2_smoke_baselines['player_id'].n_unique() == 2

h2_early_preflight_passed = True
print('✅ H2 early preflight passed before pair/evidence CV.')


✅ H2 early preflight passed before pair/evidence CV.


## 7. Source-EDA hand-level verification

These checks use the *actual engineered V2 table* and the supplied development evidence. They are deliberately broad, directional checks — not hard-coded expected means.

In [10]:
# ============================================================
# 7. VERIFY HAND-LEVEL FEATURES AGAINST FINAL EDA
# ============================================================

positive_info = dev_labels.filter(pl.col('label') == 1).select(['pair_id','behavior_family'])
evidence_keys = dev_evidence.select(['pair_id','hand_id']).unique().with_columns(pl.lit(1, dtype=pl.Int8).alias('is_evidence'))

verify_hand = (
    pl.scan_parquet(DEV_HAND_FEATURES_PATH)
    .join(positive_info.lazy(), on='pair_id')
    .join(evidence_keys.lazy(), on=['pair_id','hand_id'], how='left')
    .with_columns(pl.col('is_evidence').fill_null(0))
    .group_by(['behavior_family','is_evidence'])
    .agg(
        pl.len().alias('n'),
        pl.col('opposite_net_sign').mean().alias('opposite_net_sign'),
        pl.col('pair_river_checks').mean().alias('pair_river_checks'),
        pl.col('immediate_outsider_folds_after_aggression').mean().alias('iso_immediate_fold'),
        pl.col('net_gap_bb').mean().alias('net_gap_bb'),
        pl.col('pair_raises').mean().alias('pair_raises')
    )
    .collect()
    .sort(['behavior_family','is_evidence'])
)

display(verify_hand)

vh = verify_hand.to_pandas().set_index(['behavior_family','is_evidence'])
assert vh.loc[('directed_transfer',1),'opposite_net_sign'] > vh.loc[('directed_transfer',0),'opposite_net_sign'] + 0.50, \
    'Directed-transfer opposite-net-sign fingerprint was not reproduced.'
assert vh.loc[('soft_play',1),'pair_river_checks'] > vh.loc[('soft_play',0),'pair_river_checks'] + 0.20, \
    'Soft-play river-check fingerprint was not reproduced.'
assert vh.loc[('coordinated_isolation',1),'iso_immediate_fold'] > vh.loc[('coordinated_isolation',0),'iso_immediate_fold'] + 0.25, \
    'Coordinated-isolation immediate-outsider-fold motif was not reproduced.'

print('✅ Hand-level V2 features reproduce the final EDA fingerprints.')

behavior_family,is_evidence,n,opposite_net_sign,pair_river_checks,iso_immediate_fold,net_gap_bb,pair_raises
str,i8,u32,f64,f64,f64,f32,f64
"""coordinated_isolation""",0,10820,0.213401,0.034288,0.393438,11.10402,0.573475
"""coordinated_isolation""",1,460,0.43913,0.041304,1.0,20.898912,1.563043
"""directed_transfer""",0,17425,0.189326,0.035524,0.355007,16.894098,0.499799
"""directed_transfer""",1,725,1.0,0.074483,0.762759,60.231998,1.231724
"""soft_play""",0,15067,0.189487,0.083693,0.274242,10.513656,0.306232
"""soft_play""",1,632,0.837025,0.560127,0.511076,31.123417,0.748418


✅ Hand-level V2 features reproduce the final EDA fingerprints.


## 8. Pair aggregation: extremes, episodes, baselines, pool-relative features

Because evidence is sparse, V2 emphasizes **max / p95 / p99 / top-3 / top-5** and chronological bin maxima.

Counterparty-relative features are symmetrized across the two pair members to avoid arbitrary ID-order dependence.

In [11]:
# ============================================================
# 8. ADVANCED PAIR AGGREGATION
# ============================================================

players_lf = players

hand_schema = pl.scan_parquet(DEV_HAND_FEATURES_PATH).collect_schema()
# Keep phase_progress only for episode/time-bin construction; do not feed absolute timeline
# position directly to the pair/evidence models (final EDA flagged this as a risky shortcut).
skip_hand = {'pair_id','hand_id','table_id','phase_progress'}
HAND_FEATURES = [
    c for c, dtype in hand_schema.items()
    if dtype.is_numeric() and c not in skip_hand
]

SCORE_FEATURES = [c for c in HAND_FEATURES if c.endswith('_score_v2')]
TAIL_TOKENS = (
    'score_v2','net_gap','contribution','transfer','amount','raise','river','headsup','hu_',
    'partner','outsider','opposite','overbet','pocket_pair'
)
TAIL_FEATURES = list(dict.fromkeys(
    SCORE_FEATURES + [c for c in HAND_FEATURES if any(t in c.lower() for t in TAIL_TOKENS)]
))[:40]

score_q95 = pl.scan_parquet(DEV_HAND_FEATURES_PATH).select([
    pl.col(c).quantile(0.95).alias(c) for c in SCORE_FEATURES
]).collect().row(0, named=True)
score_q99 = pl.scan_parquet(DEV_HAND_FEATURES_PATH).select([
    pl.col(c).quantile(0.99).alias(c) for c in SCORE_FEATURES
]).collect().row(0, named=True)

RATE_NUMERATOR = {
    'fold':'folds',
    'check':'checks',
    'call':'calls',
    'bet':'bets',
    'raise':'raises',
    'aggression':'aggressive_actions',
    'headsup':'headsup_actions_raw'
}


def episode_features(path, n_bins):
    signal_cols = [c for c in [
        'directed_score_v2','soft_score_v2','isolation_score_v2','suspicious_score_v2',
        'immediate_outsider_folds_after_aggression','pair_river_checks','opposite_net_sign'
    ] if c in HAND_FEATURES]

    lf = (
        pl.scan_parquet(path)
        .with_columns(
            (pl.col('phase_progress') * n_bins).floor().clip(0,n_bins-1).cast(pl.Int8).alias('_time_bin')
        )
        .group_by(['pair_id','_time_bin'])
        .agg([
            *[pl.col(c).mean().alias(f'{c}_binmean') for c in signal_cols],
            pl.len().alias('_bin_hands')
        ])
        .group_by('pair_id')
        .agg([
            *[pl.col(f'{c}_binmean').max().alias(f'{c}_bin{n_bins}_maxmean') for c in signal_cols],
            pl.col('_bin_hands').max().alias(f'bin{n_bins}_max_hands')
        ])
    )
    return lf


def member_delta_features(path, pairs, phase):
    lf = pl.scan_parquet(path)

    agg_exprs = []
    for side in ('1','2'):
        agg_exprs.append(pl.col(f'n_actions_{side}').sum().alias(f'p{side}_pair_n_actions'))
        for _, count_col in RATE_NUMERATOR.items():
            agg_exprs.append(pl.col(f'{count_col}_{side}').sum().alias(f'p{side}_{count_col}_sum'))
        for c in ['aggressive_size_sum','aggressive_size_count','call_price_sum','call_price_count']:
            agg_exprs.append(pl.col(f'{c}_{side}').sum().alias(f'p{side}_{c}_sum'))

    member = lf.group_by('pair_id').agg(agg_exprs)

    rate_exprs = []
    for side in ('1','2'):
        denom = pl.col(f'p{side}_pair_n_actions') + 1e-6
        for metric, count_col in RATE_NUMERATOR.items():
            rate_exprs.append((pl.col(f'p{side}_{count_col}_sum') / denom).cast(pl.Float32).alias(f'p{side}_pair_{metric}_rate'))
        rate_exprs.extend([
            (pl.col(f'p{side}_aggressive_size_sum_sum') / (pl.col(f'p{side}_aggressive_size_count_sum') + 1e-6)).cast(pl.Float32).alias(f'p{side}_pair_bet_to_pot'),
            (pl.col(f'p{side}_call_price_sum_sum') / (pl.col(f'p{side}_call_price_count_sum') + 1e-6)).cast(pl.Float32).alias(f'p{side}_pair_call_price')
        ])
    member = member.with_columns(rate_exprs)

    base = pl.scan_parquet(PLAYER_ACTION_BASELINE_PATH).filter(pl.col('phase') == phase)
    base_cols = [
        'base_fold_rate','base_check_rate','base_call_rate','base_bet_rate','base_raise_rate',
        'base_aggression_rate','base_headsup_rate','base_bet_to_pot','base_call_price'
    ]
    b1 = base.select(['player_id',*base_cols]).rename({
        'player_id':'player_1', **{c:f'{c}_1' for c in base_cols}
    })
    b2 = base.select(['player_id',*base_cols]).rename({
        'player_id':'player_2', **{c:f'{c}_2' for c in base_cols}
    })

    out = (
        pairs.lazy().select(['pair_id','player_1','player_2'])
        .join(member, on='pair_id', how='left')
        .join(b1, on='player_1', how='left')
        .join(b2, on='player_2', how='left')
    )

    metric_map = {
        'fold':'base_fold_rate',
        'check':'base_check_rate',
        'call':'base_call_rate',
        'bet':'base_bet_rate',
        'raise':'base_raise_rate',
        'aggression':'base_aggression_rate',
        'headsup':'base_headsup_rate',
        'bet_to_pot':'base_bet_to_pot',
        'call_price':'base_call_price'
    }

    delta_exprs = []
    for metric, base_name in metric_map.items():
        delta_exprs.extend([
            (pl.col(f'p1_pair_{metric}_rate') - pl.col(f'{base_name}_1')).alias(f'_d1_{metric}') if metric not in ('bet_to_pot','call_price') else
            (pl.col(f'p1_pair_{metric}') - pl.col(f'{base_name}_1')).alias(f'_d1_{metric}'),
            (pl.col(f'p2_pair_{metric}_rate') - pl.col(f'{base_name}_2')).alias(f'_d2_{metric}') if metric not in ('bet_to_pot','call_price') else
            (pl.col(f'p2_pair_{metric}') - pl.col(f'{base_name}_2')).alias(f'_d2_{metric}')
        ])
    out = out.with_columns(delta_exprs)

    sym = []
    for metric in metric_map:
        d1, d2 = pl.col(f'_d1_{metric}'), pl.col(f'_d2_{metric}')
        sym.extend([
            ((d1+d2)/2).cast(pl.Float32).alias(f'delta_{metric}_mean'),
            pl.min_horizontal(d1,d2).cast(pl.Float32).alias(f'delta_{metric}_min'),
            pl.max_horizontal(d1,d2).cast(pl.Float32).alias(f'delta_{metric}_max'),
            pl.max_horizontal(d1.abs(),d2.abs()).cast(pl.Float32).alias(f'delta_{metric}_absmax'),
            (d1-d2).abs().cast(pl.Float32).alias(f'delta_{metric}_gap')
        ])
    return out.with_columns(sym).drop([f'_d{s}_{m}' for m in metric_map for s in (1,2)])


def aggregate_pair_features(path, pairs, phase, pool_path):
    lf = pl.scan_parquet(path)

    agg = [pl.len().alias('shared_hands_calc')]
    agg += [pl.col(c).mean().alias(f'{c}_mean') for c in HAND_FEATURES]
    agg += [pl.col(c).max().alias(f'{c}_max') for c in HAND_FEATURES]
    agg += [pl.col(c).quantile(0.95, interpolation='nearest').alias(f'{c}_p95') for c in TAIL_FEATURES]
    agg += [pl.col(c).quantile(0.99, interpolation='nearest').alias(f'{c}_p99') for c in TAIL_FEATURES]
    agg += [pl.col(c).top_k(3).mean().alias(f'{c}_top3') for c in TAIL_FEATURES]
    agg += [pl.col(c).top_k(5).mean().alias(f'{c}_top5') for c in TAIL_FEATURES]
    agg += [(pl.col(c) >= score_q95[c]).mean().alias(f'{c}_rate95') for c in SCORE_FEATURES]
    agg += [(pl.col(c) >= score_q99[c]).mean().alias(f'{c}_rate99') for c in SCORE_FEATURES]

    pair_stats = lf.group_by('pair_id').agg(agg)

    # Signed direction consistency / transfer asymmetry across the whole pair history.
    direction = lf.group_by('pair_id').agg(
        pl.col('signed_net_diff_bb').sum().alias('_signed_sum'),
        pl.col('signed_net_diff_bb').abs().sum().alias('_abs_signed_sum'),
        pl.col('transfer_1_to_2_bb').sum().alias('_t12_sum'),
        pl.col('transfer_2_to_1_bb').sum().alias('_t21_sum'),
        (pl.col('max_loss_bb') >= 10).mean().alias('large_loss_rate_10bb'),
        (pl.col('max_loss_bb') >= 20).mean().alias('large_loss_rate_20bb'),
        (pl.col('max_loss_bb') >= 40).mean().alias('large_loss_rate_40bb')
    ).with_columns(
        (pl.col('_signed_sum').abs() / (pl.col('_abs_signed_sum') + 1e-6)).cast(pl.Float32).alias('net_direction_consistency'),
        ((pl.col('_t12_sum')-pl.col('_t21_sum')).abs() / (pl.col('_t12_sum')+pl.col('_t21_sum')+1e-6)).cast(pl.Float32).alias('transfer_asymmetry')
    ).drop(['_signed_sum','_abs_signed_sum','_t12_sum','_t21_sum'])

    episode10 = episode_features(path,10)
    episode20 = episode_features(path,20)
    deltas = member_delta_features(path,pairs,phase)

    # Lightweight metadata features.
    meta_cols = ['player_id','account_age_days','experience_hands_bucket','preferred_stake','region_bucket','client_family']
    meta = players_lf.select(meta_cols)
    m1 = meta.rename({c:('player_1' if c == 'player_id' else f'{c}_1') for c in meta_cols})
    m2 = meta.rename({c:('player_2' if c == 'player_id' else f'{c}_2') for c in meta_cols})
    pair_meta = (
        pairs.lazy().select(['pair_id','player_1','player_2'])
        .join(m1,on='player_1',how='left')
        .join(m2,on='player_2',how='left')
        .select(
            'pair_id',
            (pl.col('account_age_days_1').cast(pl.Float32)-pl.col('account_age_days_2').cast(pl.Float32)).abs().alias('account_age_gap'),
            (pl.col('experience_hands_bucket_1') == pl.col('experience_hands_bucket_2')).cast(pl.Int8).alias('same_experience'),
            (pl.col('preferred_stake_1') == pl.col('preferred_stake_2')).cast(pl.Int8).alias('same_stake'),
            (pl.col('region_bucket_1') == pl.col('region_bucket_2')).cast(pl.Int8).alias('same_region'),
            (pl.col('client_family_1') == pl.col('client_family_2')).cast(pl.Int8).alias('same_client')
        )
    )

    pool = (
        pl.scan_parquet(pool_path)
        .rename({'p1':'player_1','p2':'player_2'})
    )

    # Evaluation preparation already carries a reconstruction-only shared_hands_calc.
    # Drop it before joining the phase-specific hand aggregation so development/evaluation
    # expose the exact same feature name/schema.
    pair_left = pairs.drop('shared_hands_calc') if 'shared_hands_calc' in pairs.columns else pairs

    out = (
        pair_left.lazy()
        .join(pair_stats,on='pair_id',how='left')
        .join(direction,on='pair_id',how='left')
        .join(episode10,on='pair_id',how='left')
        .join(episode20,on='pair_id',how='left')
        .join(deltas.drop(['player_1','player_2']),on='pair_id',how='left')
        .join(pair_meta,on='pair_id',how='left')
        .join(pool,on=['table_id','player_1','player_2'],how='left')
        .with_columns(
            pl.col('shared_hands_calc').cast(pl.Float32).log1p().alias('exposure_log1p'),
            (pl.col('shared_hands_calc') / (pl.col('shared_hands_calc') + SHRINKAGE_ALPHA)).cast(pl.Float32).alias('exposure_confidence_20')
        )
    )

    # Sparse planted events are about 4% of positive-pair hands. Shrink event rates toward
    # their global development priors so a tiny-exposure pair cannot look extreme from one hand.
    for c in SCORE_FEATURES:
        out = out.with_columns(
            ((pl.col(f'{c}_rate95') * pl.col('shared_hands_calc') + 0.05 * SHRINKAGE_ALPHA) /
             (pl.col('shared_hands_calc') + SHRINKAGE_ALPHA)).cast(pl.Float32).alias(f'{c}_rate95_smoothed'),
            ((pl.col(f'{c}_rate99') * pl.col('shared_hands_calc') + 0.01 * SHRINKAGE_ALPHA) /
             (pl.col('shared_hands_calc') + SHRINKAGE_ALPHA)).cast(pl.Float32).alias(f'{c}_rate99_smoothed')
        )

    shrink_delta_cols = [
        'delta_fold_mean','delta_check_mean','delta_call_mean','delta_bet_mean','delta_raise_mean',
        'delta_aggression_mean','delta_headsup_mean','delta_bet_to_pot_mean','delta_call_price_mean'
    ]
    out = out.with_columns([
        (pl.col(c) * pl.col('exposure_confidence_20')).cast(pl.Float32).alias(f'{c}_shrunk')
        for c in shrink_delta_cols
    ])

    return out.collect(engine='streaming')

with tqdm(total=2, desc='Aggregate V2 pair features') as pbar:
    dev_features = aggregate_pair_features(DEV_HAND_FEATURES_PATH,dev_pairs_prep,'development',DEV_POOL_PATH)
    pbar.update()
    eval_features = aggregate_pair_features(EVAL_HAND_FEATURES_PATH,eval_pairs_prep,'evaluation',EVAL_POOL_PATH)
    pbar.update()

print('Development feature matrix:', dev_features.shape)
print('Evaluation feature matrix: ', eval_features.shape)
print('Hand numeric features:', len(HAND_FEATURES), '| tail/extreme features:', len(TAIL_FEATURES))

Aggregate V2 pair features:   0%|          | 0/2 [00:00<?, ?it/s]

Development feature matrix: (30660, 502)
Evaluation feature matrix:  (112540, 501)
Hand numeric features: 76 | tail/extreme features: 40


## 9. Source-EDA pair-level verification

This verifies the most important EDA directions on the *actual V2 pair matrix* before any model is trained.

In [12]:
# ============================================================
# 9. VERIFY PAIR FEATURES AGAINST FINAL EDA
# ============================================================

known_verify = dev_features.filter(pl.col('is_labeled') == True).select([
    'label','behavior_family','delta_check_mean','delta_call_mean','delta_raise_mean','delta_aggression_mean',
    'delta_bet_to_pot_mean','pool_pct_net_gap','pool_pct_combined_contrib'
])

verify_pair = known_verify.group_by(['label','behavior_family']).agg([
    pl.col(c).mean().alias(c) for c in [
        'delta_check_mean','delta_call_mean','delta_raise_mean','delta_aggression_mean',
        'delta_bet_to_pot_mean','pool_pct_net_gap','pool_pct_combined_contrib'
    ]
]).sort(['label','behavior_family'])

display(verify_pair)
vp = verify_pair.to_pandas().set_index(['label','behavior_family'])
neg = vp.loc[(0,'none')]
soft = vp.loc[(1,'soft_play')]
iso = vp.loc[(1,'coordinated_isolation')]
directed = vp.loc[(1,'directed_transfer')]

assert soft['delta_check_mean'] > neg['delta_check_mean'] + 0.02, 'Soft-play check delta does not match EDA.'
assert soft['delta_raise_mean'] < neg['delta_raise_mean'] - 0.01, 'Soft-play raise suppression does not match EDA.'
assert iso['delta_raise_mean'] > neg['delta_raise_mean'] + 0.02, 'Isolation raise delta does not match EDA.'
assert directed['pool_pct_net_gap'] > neg['pool_pct_net_gap'] + 0.20, 'Directed-transfer pool net-gap percentile does not match EDA.'

print('✅ Pair-level V2 features reproduce the final EDA directions.')

label,behavior_family,delta_check_mean,delta_call_mean,delta_raise_mean,delta_aggression_mean,delta_bet_to_pot_mean,pool_pct_net_gap,pool_pct_combined_contrib
i64,str,f32,f32,f32,f32,f32,f64,f64
0,"""none""",-0.000958,-0.001352,0.000152,-0.001233,0.00443,0.493192,0.490981
1,"""coordinated_isolation""",-0.002587,-0.014875,0.052885,0.053316,0.060766,0.676073,0.702222
1,"""directed_transfer""",-0.00076,0.03239,0.01636,0.045761,0.017909,0.897711,0.908037
1,"""soft_play""",0.052334,0.085875,-0.036317,-0.036398,-0.092739,0.681304,0.681047


✅ Pair-level V2 features reproduce the final EDA directions.


## 10. Strict outer CV for pair risk and behavior

Every outer validation fold is now untouched by model selection:

1. choose boosting rounds on an **inner stop split** drawn only from outer-training rows;
2. retrain on the complete outer-training fold with that fixed number of trees;
3. predict the outer validation fold.

Unknown sampling is label-independent. Its lower positive-player-control weight is assigned fold-by-fold using **outer-training positives only**.


In [13]:
# ============================================================
# 10A. BUILD TRAIN MATRIX / FOLDS / STRICT HELPERS
# ============================================================

behavior_names = np.array(['directed_transfer','soft_play','coordinated_isolation'])

train_df = dev_features.sort('pair_id')
eval_features = eval_features.sort('pair_id')

known = train_df['is_labeled'].to_numpy().astype(bool)
y = train_df['label'].fill_null(0).to_numpy().astype(np.int8)
behavior_y = train_df['behavior_id'].to_numpy().astype(np.int8)
pair_ids_np = np.array(train_df['pair_id'].to_list(), dtype=object)
player1_np = np.array(train_df['player_1'].to_list(), dtype=object)
player2_np = np.array(train_df['player_2'].to_list(), dtype=object)
population_weight = train_df['population_weight'].to_numpy().astype(np.float64)

exclude_features = {
    'pair_id','player_1','player_2','table_id','label','behavior_id',
    'is_labeled','is_pu','population_weight'
}
feature_cols = [
    c for c,dtype in train_df.schema.items()
    if dtype.is_numeric() and c not in exclude_features
]

X = (
    train_df.select(feature_cols).to_pandas()
    .replace([np.inf,-np.inf],np.nan)
    .fillna(0)
    .astype(np.float32)
)

# 0 = known negative, 1-3 positive families, 4 = label-independent sampled unknown.
strata = behavior_y.copy().astype(np.int16)
strata[~known] = 4

print('Training rows:', len(train_df))
print('Features:', len(feature_cols))
print('Strata counts:', dict(zip(*np.unique(strata, return_counts=True))))

primary_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_id = np.full(len(train_df), -1, dtype=np.int8)
for fold,(tr,va) in enumerate(primary_cv.split(X,strata)):
    fold_id[va] = fold
assert (fold_id >= 0).all()


def _positive_players(indices):
    indices = np.asarray(indices, dtype=int)
    pos = indices[known[indices] & (y[indices] == 1)]
    if len(pos) == 0:
        return set()
    return set(player1_np[pos]).union(set(player2_np[pos]))


def make_fit_weight(indices, positive_source_indices):
    """Weights for rows in `indices`; hard-PU status uses ONLY positive_source_indices labels."""
    indices = np.asarray(indices, dtype=int)
    w = np.ones(len(indices), dtype=np.float32)
    local_known = known[indices]
    local_y = y[indices]

    w[(local_known) & (local_y == 1)] = POSITIVE_FIT_WEIGHT
    w[~local_known] = UNKNOWN_BASE_WEIGHT

    pos_players = _positive_players(positive_source_indices)
    if pos_players:
        hard = (~local_known) & (
            np.isin(player1_np[indices], list(pos_players)) |
            np.isin(player2_np[indices], list(pos_players))
        )
        w[hard] = UNKNOWN_POSITIVE_PLAYER_WEIGHT
    return w


def stratified_inner_split(indices, stratum_values, seed, requested_splits=4):
    indices = np.asarray(indices, dtype=int)
    values = np.asarray(stratum_values)
    _, counts = np.unique(values, return_counts=True)
    n_splits = int(min(requested_splits, counts.min()))
    if n_splits < 2:
        raise RuntimeError('Not enough examples for strict stratified inner split.')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fit_rel, stop_rel = next(cv.split(np.zeros(len(indices)), values))
    return indices[fit_rel], indices[stop_rel]


def stratified_inner_three_way(indices, stratum_values, seed, requested_splits=5):
    """Disjoint FIT / EARLY-STOP / ROUTING-TUNE split inside one outer-training fold."""
    indices = np.asarray(indices, dtype=int)
    values = np.asarray(stratum_values)
    _, counts = np.unique(values, return_counts=True)
    n_splits = int(min(requested_splits, counts.min()))
    if n_splits < 3:
        raise RuntimeError('Not enough examples for strict three-way inner split.')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    validation_folds = [va for _,va in cv.split(np.zeros(len(indices)), values)]
    stop_rel = validation_folds[0]
    tune_rel = validation_folds[1]
    fit_rel = np.concatenate(validation_folds[2:])
    return indices[np.sort(fit_rel)], indices[np.sort(stop_rel)], indices[np.sort(tune_rel)]


def _weighted_average_precision_stable(y_true, scores, weights):
    """PU diagnostic only; NOT the official competition metric."""
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=float)
    weights = np.asarray(weights, dtype=float)
    pos_weight = float(np.sum(weights * y_true))
    if pos_weight <= 0:
        return 0.0
    order = np.argsort(-scores, kind='mergesort')
    yy = y_true[order]
    ww = weights[order]
    weighted_pos = yy * ww
    cum_pos = np.cumsum(weighted_pos)
    cum_mass = np.cumsum(ww)
    precision = cum_pos / np.maximum(cum_mass, 1e-12)
    return float(np.sum(precision * weighted_pos) / pos_weight)


def weighted_top_mask(scores, weights, rate):
    """Select a fixed fraction of estimated population mass, stable under score ties."""
    scores = np.asarray(scores, dtype=float)
    weights = np.asarray(weights, dtype=float)
    assert len(scores) == len(weights)
    if len(scores) == 0:
        return np.zeros(0, dtype=bool)
    target_mass = float(np.clip(rate,0,1) * weights.sum())
    order = np.argsort(-scores, kind='mergesort')
    cumulative = np.cumsum(weights[order])
    n = int(np.searchsorted(cumulative, target_mass, side='left') + 1)
    n = max(1, min(n, len(scores)))
    mask = np.zeros(len(scores), dtype=bool)
    mask[order[:n]] = True
    return mask


def _behavior_activation_score(risk_scores, behavior_proba):
    risk_scores = np.asarray(risk_scores, dtype=float)
    behavior_proba = np.asarray(behavior_proba, dtype=float)
    if BEHAVIOR_ACTIVATION_MODE == 'risk_x_confidence':
        return risk_scores * np.clip(behavior_proba.max(axis=1),0.05,1.0)
    return risk_scores


def _weighted_behavior_map_on_subset(indices, risk_scores, behavior_proba, active):
    """PU population-weighted routing surrogate; not the official competition metric."""
    indices = np.asarray(indices, dtype=int)
    family = np.asarray(behavior_proba).argmax(axis=1)
    vals = []
    for k in range(3):
        truth = (behavior_y[indices] == k+1).astype(np.int8)
        score = np.where(np.asarray(active) & (family == k), np.asarray(risk_scores), 0.0)
        vals.append(_weighted_average_precision_stable(truth,score,population_weight[indices]))
    return float(np.mean(vals)), vals


def select_nested_behavior_rate(indices, risk_scores, behavior_proba):
    """Select rate only on an inner tuning split; ties prefer the smaller rate."""
    indices = np.asarray(indices,dtype=int)
    activation = _behavior_activation_score(risk_scores,behavior_proba)
    rows = []
    for rate in BEHAVIOR_RATE_GRID:
        active = weighted_top_mask(activation,population_weight[indices],float(rate))
        score,_ = _weighted_behavior_map_on_subset(indices,risk_scores,behavior_proba,active)
        selected_mass = float(population_weight[indices][active].sum()/population_weight[indices].sum())
        rows.append((float(rate),float(score),selected_mass))
    # Grid is ascending, so np.argmax deterministically picks the smallest rate on exact ties.
    best_i = int(np.argmax([r[1] for r in rows]))
    return rows[best_i], rows


# Population weights approximate the full eligible development unknown population.
print(f'Estimated sampled development population mass: {population_weight.sum():,.1f}')
print('✅ Strict CV helpers ready.')


Training rows: 30660
Features: 490
Strata counts: {np.int16(0): np.int64(1488), np.int16(1): np.int64(148), np.int16(2): np.int64(132), np.int16(3): np.int64(92), np.int16(4): np.int64(28800)}
Estimated sampled development population mass: 137,739.0
✅ Strict CV helpers ready.


## 10B. Pseudo-future temporal-transfer diagnostic

This remains a reduced-feature diagnostic, not the primary score. The outer late-period rows are now untouched by early stopping: boosting rounds are selected on an inner outer-training stop split and then the model is retrained before outer-late prediction.


In [14]:
# ============================================================
# 10B. STRICT PSEUDO-FUTURE EARLY -> LATE DIAGNOSTIC
# ============================================================

TEMPORAL_SIGNAL_COLS = [c for c in [
    'directed_score_v2','soft_score_v2','isolation_score_v2','suspicious_score_v2',
    'net_gap_bb','pair_contribution_bb','opposite_net_sign','pair_river_checks',
    'immediate_outsider_folds_after_aggression','pair_raises','pair_calls','pair_checks',
    'pair_overbets','max_amount_bb'
] if c in HAND_FEATURES]


def temporal_snapshot(progress_filter):
    lf = pl.scan_parquet(DEV_HAND_FEATURES_PATH).filter(progress_filter)
    exprs = [pl.len().cast(pl.Int32).alias('snapshot_shared_hands')]
    for c in TEMPORAL_SIGNAL_COLS:
        exprs.extend([
            pl.col(c).mean().alias(f'{c}_mean'),
            pl.col(c).max().alias(f'{c}_max'),
            pl.col(c).quantile(0.95, interpolation='nearest').alias(f'{c}_p95'),
            pl.col(c).top_k(3).mean().alias(f'{c}_top3')
        ])
    return lf.group_by('pair_id').agg(exprs).collect(engine='streaming')


temporal_summary = None
if RUN_TEMPORAL_DIAGNOSTIC:
    early = temporal_snapshot(pl.col('phase_progress') <= 0.70)
    late = temporal_snapshot(pl.col('phase_progress') > 0.70)

    temporal_cols = [c for c in early.columns if c != 'pair_id']
    early_aligned = train_df.select(['pair_id']).join(early,on='pair_id',how='left').sort('pair_id')
    late_aligned = train_df.select(['pair_id']).join(late,on='pair_id',how='left').sort('pair_id')
    assert early_aligned['pair_id'].to_list() == train_df['pair_id'].to_list()
    assert late_aligned['pair_id'].to_list() == train_df['pair_id'].to_list()

    X_early = early_aligned.select(temporal_cols).to_pandas().replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
    X_late = late_aligned.select(temporal_cols).to_pandas().replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)

    temporal_oof = np.zeros(len(train_df),dtype=np.float32)
    temporal_seen = np.zeros(len(train_df),dtype=bool)
    temporal_rounds = []

    temporal_base = dict(
        learning_rate=0.03,max_depth=4,min_child_weight=6,
        subsample=0.88,colsample_bytree=0.85,reg_alpha=0.15,reg_lambda=6.0,
        objective='binary:logistic',eval_metric='aucpr',tree_method='hist',n_jobs=-1
    )

    for fold in tqdm(range(N_FOLDS),desc='Strict pseudo-future CV'):
        outer_tr = np.flatnonzero(fold_id != fold)
        outer_va = np.flatnonzero(fold_id == fold)
        inner_fit, inner_stop = stratified_inner_split(
            outer_tr, strata[outer_tr], SEED+700+fold, requested_splits=4
        )

        probe = XGBClassifier(
            **temporal_base,n_estimators=900,early_stopping_rounds=70,
            random_state=SEED+700+fold
        )
        probe.fit(
            X_early.iloc[inner_fit],y[inner_fit],
            sample_weight=make_fit_weight(inner_fit,inner_fit),
            eval_set=[(X_late.iloc[inner_stop],y[inner_stop])],
            sample_weight_eval_set=[population_weight[inner_stop]],
            verbose=False
        )
        best_round = max(1,int(probe.best_iteration)+1)
        temporal_rounds.append(best_round)

        model = XGBClassifier(
            **temporal_base,n_estimators=best_round,random_state=SEED+1700+fold
        )
        model.fit(
            X_early.iloc[outer_tr],y[outer_tr],
            sample_weight=make_fit_weight(outer_tr,outer_tr),verbose=False
        )
        temporal_oof[outer_va] = model.predict_proba(X_late.iloc[outer_va])[:,1]
        temporal_seen[outer_va] = True

    kmask = known & temporal_seen
    temporal_labeled_ap = _average_precision_exact(y[kmask],temporal_oof[kmask])
    temporal_stress_ap = _weighted_average_precision_stable(
        y[temporal_seen],temporal_oof[temporal_seen],population_weight[temporal_seen]
    )
    temporal_summary = {
        'early_fraction':0.70,
        'exact_labeled_pair_ap':float(temporal_labeled_ap),
        'pu_weighted_stable_ap_surrogate':float(temporal_stress_ap),
        'n_features':len(temporal_cols),
        'median_rounds':int(np.median(temporal_rounds))
    }
    print(json.dumps(temporal_summary,indent=2))
else:
    print('Skipped pseudo-future diagnostic.')


Strict pseudo-future CV:   0%|          | 0/5 [00:00<?, ?it/s]

{
  "early_fraction": 0.7,
  "exact_labeled_pair_ap": 0.616815802205983,
  "pu_weighted_stable_ap_surrogate": 0.08859276894721384,
  "n_features": 57,
  "median_rounds": 148
}


In [15]:
# ============================================================
# 10C. STRICT OUTER OOF: RISK + BEHAVIOR + NESTED ROUTING RATE
# ============================================================

risk_base_params = dict(
    learning_rate=0.022,max_depth=5,min_child_weight=7,
    subsample=0.88,colsample_bytree=0.80,reg_alpha=0.18,reg_lambda=7.0,
    objective='binary:logistic',eval_metric='aucpr',tree_method='hist',n_jobs=-1
)

behavior_base_params = dict(
    learning_rate=0.028,max_depth=3,min_child_weight=3,
    subsample=0.88,colsample_bytree=0.82,reg_alpha=0.10,reg_lambda=5.0,
    objective='multi:softprob',num_class=3,eval_metric='mlogloss',tree_method='hist',n_jobs=-1
)

oof_risk = np.zeros(len(train_df),dtype=np.float32)
oof_behavior = np.zeros((len(train_df),3),dtype=np.float32)
behavior_active_oof = np.zeros(len(train_df),dtype=bool)

risk_models = []
behavior_models = []
risk_best_rounds = []
behavior_best_rounds = []
behavior_rate_by_fold = []
behavior_rate_tune_score_by_fold = []
behavior_rate_mass_by_fold = []
pair_cv_manifest = []

for fold in tqdm(range(N_FOLDS),desc='Strict nested primary pair CV'):
    outer_va = np.flatnonzero(fold_id == fold)
    outer_tr = np.flatnonzero(fold_id != fold)

    # Three disjoint inner partitions: fit, early-stop, routing-tune.
    inner_fit,inner_stop,inner_tune = stratified_inner_three_way(
        outer_tr,strata[outer_tr],SEED+10+fold,requested_splits=5
    )

    fit_set=set(inner_fit.tolist()); stop_set=set(inner_stop.tolist()); tune_set=set(inner_tune.tolist())
    outer_va_set=set(outer_va.tolist()); outer_tr_set=set(outer_tr.tolist())
    inner_disjoint=(fit_set.isdisjoint(stop_set) and fit_set.isdisjoint(tune_set) and stop_set.isdisjoint(tune_set))
    inner_inside_outer=(fit_set|stop_set|tune_set).issubset(outer_tr_set)
    outer_disjoint=outer_tr_set.isdisjoint(outer_va_set)
    assert inner_disjoint and inner_inside_outer and outer_disjoint

    # ----- Risk: early stopping only on inner_stop.
    risk_probe = XGBClassifier(
        **risk_base_params,n_estimators=1700,early_stopping_rounds=110,
        random_state=SEED+fold
    )
    risk_probe.fit(
        X.iloc[inner_fit],y[inner_fit],
        sample_weight=make_fit_weight(inner_fit,inner_fit),
        eval_set=[(X.iloc[inner_stop],y[inner_stop])],
        sample_weight_eval_set=[population_weight[inner_stop]],
        verbose=False
    )
    risk_rounds=max(1,int(risk_probe.best_iteration)+1)
    risk_best_rounds.append(risk_rounds)

    # Retrain on fit+stop with fixed rounds; predict disjoint routing-tune rows.
    risk_tune_train=np.sort(np.concatenate([inner_fit,inner_stop]))
    risk_tune_model=XGBClassifier(
        **risk_base_params,n_estimators=risk_rounds,random_state=SEED+500+fold
    )
    risk_tune_model.fit(
        X.iloc[risk_tune_train],y[risk_tune_train],
        sample_weight=make_fit_weight(risk_tune_train,risk_tune_train),verbose=False
    )
    risk_tune_pred=risk_tune_model.predict_proba(X.iloc[inner_tune])[:,1]

    # ----- Behavior: same fit/stop/tune separation, positives only for fitting.
    beh_fit=inner_fit[y[inner_fit] == 1]
    beh_stop=inner_stop[y[inner_stop] == 1]
    beh_tune_train=np.sort(np.concatenate([beh_fit,beh_stop]))
    assert len(np.unique(behavior_y[beh_fit])) == 3 and len(np.unique(behavior_y[beh_stop])) == 3

    counts_fit=np.bincount(behavior_y[beh_fit]-1,minlength=3)
    cw_fit=len(beh_fit)/(3*np.maximum(counts_fit,1))
    behavior_probe=XGBClassifier(
        **behavior_base_params,n_estimators=850,early_stopping_rounds=80,
        random_state=SEED+100+fold
    )
    behavior_probe.fit(
        X.iloc[beh_fit],behavior_y[beh_fit]-1,
        sample_weight=cw_fit[behavior_y[beh_fit]-1],
        eval_set=[(X.iloc[beh_stop],behavior_y[beh_stop]-1)],verbose=False
    )
    behavior_rounds=max(1,int(behavior_probe.best_iteration)+1)
    behavior_best_rounds.append(behavior_rounds)

    counts_tune_train=np.bincount(behavior_y[beh_tune_train]-1,minlength=3)
    cw_tune_train=len(beh_tune_train)/(3*np.maximum(counts_tune_train,1))
    behavior_tune_model=XGBClassifier(
        **behavior_base_params,n_estimators=behavior_rounds,random_state=SEED+600+fold
    )
    behavior_tune_model.fit(
        X.iloc[beh_tune_train],behavior_y[beh_tune_train]-1,
        sample_weight=cw_tune_train[behavior_y[beh_tune_train]-1],verbose=False
    )
    behavior_tune_proba=behavior_tune_model.predict_proba(X.iloc[inner_tune])

    (selected_rate,selected_tune_score,selected_tune_mass),rate_table = select_nested_behavior_rate(
        inner_tune,risk_tune_pred,behavior_tune_proba
    )
    behavior_rate_by_fold.append(selected_rate)
    behavior_rate_tune_score_by_fold.append(selected_tune_score)

    # ----- Final outer-fold risk model: all outer-train, fixed rounds.
    risk_model=XGBClassifier(
        **risk_base_params,n_estimators=risk_rounds,random_state=SEED+1000+fold
    )
    risk_model.fit(
        X.iloc[outer_tr],y[outer_tr],
        sample_weight=make_fit_weight(outer_tr,outer_tr),verbose=False
    )
    oof_risk[outer_va]=risk_model.predict_proba(X.iloc[outer_va])[:,1]
    risk_models.append(risk_model)

    # ----- Final outer-fold behavior model: all outer-train positives, fixed rounds.
    pos_outer_tr=outer_tr[y[outer_tr] == 1]
    counts_outer=np.bincount(behavior_y[pos_outer_tr]-1,minlength=3)
    cw_outer=len(pos_outer_tr)/(3*np.maximum(counts_outer,1))
    behavior_model=XGBClassifier(
        **behavior_base_params,n_estimators=behavior_rounds,random_state=SEED+1100+fold
    )
    behavior_model.fit(
        X.iloc[pos_outer_tr],behavior_y[pos_outer_tr]-1,
        sample_weight=cw_outer[behavior_y[pos_outer_tr]-1],verbose=False
    )
    oof_behavior[outer_va]=behavior_model.predict_proba(X.iloc[outer_va])
    behavior_models.append(behavior_model)

    outer_activation=_behavior_activation_score(oof_risk[outer_va],oof_behavior[outer_va])
    outer_active=weighted_top_mask(
        outer_activation,population_weight[outer_va],selected_rate
    )
    behavior_active_oof[outer_va]=outer_active
    outer_selected_mass=float(population_weight[outer_va][outer_active].sum()/population_weight[outer_va].sum())
    behavior_rate_mass_by_fold.append(outer_selected_mass)

    max_outer_row_mass=float(population_weight[outer_va].max()/population_weight[outer_va].sum())
    pair_cv_manifest.append({
        'fold':int(fold),
        'outer_train_n':int(len(outer_tr)),
        'outer_validation_n':int(len(outer_va)),
        'inner_fit_n':int(len(inner_fit)),
        'inner_stop_n':int(len(inner_stop)),
        'inner_tune_n':int(len(inner_tune)),
        'outer_train_validation_disjoint':bool(outer_disjoint),
        'inner_fit_stop_tune_disjoint':bool(inner_disjoint),
        'inner_subsets_inside_outer_train':bool(inner_inside_outer),
        'risk_stop_excludes_outer_validation':bool(set(inner_stop.tolist()).isdisjoint(outer_va_set)),
        'behavior_stop_excludes_outer_validation':bool(set(beh_stop.tolist()).isdisjoint(outer_va_set)),
        'routing_tune_excludes_outer_validation':bool(set(inner_tune.tolist()).isdisjoint(outer_va_set)),
        'positive_weight_source_outer_train_only':bool(set(outer_tr.tolist()).isdisjoint(outer_va_set)),
        'selected_behavior_rate':float(selected_rate),
        'routing_tune_score':float(selected_tune_score),
        'routing_tune_mass':float(selected_tune_mass),
        'outer_selected_mass':float(outer_selected_mass),
        'outer_max_single_row_mass':float(max_outer_row_mass),
        'risk_trees':int(risk_rounds),
        'behavior_trees':int(behavior_rounds),
    })

    known_va=outer_va[known[outer_va]]
    fold_exact_ap=_average_precision_exact(y[known_va],oof_risk[known_va])
    fold_pu_surrogate=_weighted_average_precision_stable(
        y[outer_va],oof_risk[outer_va],population_weight[outer_va]
    )
    print(
        f'Fold {fold+1}: exact labeled AP={fold_exact_ap:.4f} | '
        f'PU weighted-stable={fold_pu_surrogate:.4f} | '
        f'behavior rate={selected_rate:.3%} (outer mass={outer_selected_mass:.3%}) | '
        f'risk trees={risk_rounds} | behavior trees={behavior_rounds}'
    )

assert behavior_active_oof.dtype == bool and len(behavior_active_oof) == len(train_df)
primary_exact_pair_ap=_average_precision_exact(y[known],oof_risk[known])
primary_pu_surrogate=_weighted_average_precision_stable(y,oof_risk,population_weight)

print('\nStrict OOF exact labeled Pair AP:',primary_exact_pair_ap)
print('Strict OOF PU weighted-stable AP surrogate:',primary_pu_surrogate)
print('Nested behavior rates:',[round(r,5) for r in behavior_rate_by_fold])
print('Median risk trees:',int(np.median(risk_best_rounds)))
print('Median behavior trees:',int(np.median(behavior_best_rounds)))


Strict nested primary pair CV:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 1: exact labeled AP=0.9744 | PU weighted-stable=0.5644 | behavior rate=1.000% (outer mass=1.010%) | risk trees=1038 | behavior trees=849
Fold 2: exact labeled AP=0.9658 | PU weighted-stable=0.6775 | behavior rate=1.500% (outer mass=1.515%) | risk trees=748 | behavior trees=551
Fold 3: exact labeled AP=0.9764 | PU weighted-stable=0.5868 | behavior rate=2.000% (outer mass=2.008%) | risk trees=201 | behavior trees=292
Fold 4: exact labeled AP=0.9853 | PU weighted-stable=0.6074 | behavior rate=1.000% (outer mass=1.000%) | risk trees=233 | behavior trees=362
Fold 5: exact labeled AP=0.9828 | PU weighted-stable=0.5234 | behavior rate=2.000% (outer mass=2.011%) | risk trees=1372 | behavior trees=850

Strict OOF exact labeled Pair AP: 0.9737033653053857
Strict OOF PU weighted-stable AP surrogate: 0.567863434551682
Nested behavior rates: [0.01, 0.015, 0.02, 0.01, 0.02]
Median risk trees: 748
Median behavior trees: 551


In [16]:
# ============================================================
# 10D. OPTIONAL STRICT TABLE-GROUP ROBUSTNESS CV (RISK ONLY)
# ============================================================

if RUN_ROBUST_GROUP_CV:
    groups = np.array(train_df['table_id'].to_list(),dtype=object)
    group_cv = StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)
    robust_oof = np.zeros(len(train_df),dtype=np.float32)
    robust_seen = np.zeros(len(train_df),dtype=bool)

    for fold,(outer_tr,outer_va) in enumerate(group_cv.split(X,strata,groups)):
        inner_fit,inner_stop = stratified_inner_split(
            outer_tr,strata[outer_tr],SEED+500+fold,requested_splits=4
        )
        probe = XGBClassifier(
            **risk_base_params,n_estimators=1100,early_stopping_rounds=80,
            random_state=SEED+500+fold
        )
        probe.fit(
            X.iloc[inner_fit],y[inner_fit],
            sample_weight=make_fit_weight(inner_fit,inner_fit),
            eval_set=[(X.iloc[inner_stop],y[inner_stop])],
            sample_weight_eval_set=[population_weight[inner_stop]],verbose=False
        )
        rounds = max(1,int(probe.best_iteration)+1)
        model = XGBClassifier(
            **risk_base_params,n_estimators=rounds,random_state=SEED+1500+fold
        )
        model.fit(
            X.iloc[outer_tr],y[outer_tr],
            sample_weight=make_fit_weight(outer_tr,outer_tr),verbose=False
        )
        robust_oof[outer_va] = model.predict_proba(X.iloc[outer_va])[:,1]
        robust_seen[outer_va] = True
        kva = outer_va[known[outer_va]]
        print(f'Group fold {fold+1}: exact labeled AP={_average_precision_exact(y[kva],robust_oof[kva]):.4f}')

    mask = known & robust_seen
    print('Strict table-group exact labeled Pair AP:',_average_precision_exact(y[mask],robust_oof[mask]))
else:
    print('Skipped optional table-group CV.')


Skipped optional table-group CV.


## 11. Nested behavior activation + exact metric reporting

The earlier 5% activation rate was itself selected on the disclosed development labels in the 0.55758 source notebook, and V2.1 also changed its meaning from row quota to estimated population-mass quota.

V2.2 therefore does **not** treat 5% as an independent constant. For each outer fold:

1. outer-training rows are split into disjoint inner **fit / early-stop / routing-tune** partitions;
2. risk and behavior models predict the inner routing-tune partition without training on it;
3. the population-weighted activation rate is selected only on that inner routing-tune partition;
4. the selected rate is applied to the untouched outer-validation fold.

The final evaluation rate is the median of the five outer-fold selected rates. The tuning objective remains explicitly labeled a PU population-weighted surrogate; exact behavior AP reporting uses the supplied stable AP helper.


In [17]:
# ============================================================
# 11. NESTED BEHAVIOR ACTIVATION REPORTING
# ============================================================

family_oof=oof_behavior.argmax(axis=1)
family_conf_oof=oof_behavior.max(axis=1)


def behavior_map_exact(active,row_mask):
    row_mask=np.asarray(row_mask,dtype=bool)
    vals=[]
    for k in range(3):
        truth=(behavior_y[row_mask] == k+1).astype(np.int8)
        score=np.where(
            active[row_mask] & (family_oof[row_mask] == k),
            oof_risk[row_mask],0.0
        )
        vals.append(_average_precision_exact(truth,score))
    return float(np.mean(vals)),vals


def behavior_map_pu_surrogate(active):
    vals=[]
    for k in range(3):
        truth=(behavior_y == k+1).astype(np.int8)
        score=np.where(active & (family_oof == k),oof_risk,0.0)
        vals.append(_weighted_average_precision_stable(truth,score,population_weight))
    return float(np.mean(vals)),vals

behavior_map_labeled,behavior_ap_labeled=behavior_map_exact(behavior_active_oof,known)
behavior_map_stress,behavior_ap_stress=behavior_map_pu_surrogate(behavior_active_oof)
family_accuracy=(family_oof[known & (y == 1)] + 1 == behavior_y[known & (y == 1)]).mean()

selected_population_mass=float(population_weight[behavior_active_oof].sum()/population_weight.sum())
FINAL_BEHAVIOR_POSITIVE_RATE=float(np.median(np.asarray(behavior_rate_by_fold,dtype=float)))
assert FINAL_BEHAVIOR_POSITIVE_RATE in set(BEHAVIOR_RATE_GRID.tolist())

print('Activation mode:',BEHAVIOR_ACTIVATION_MODE)
print('Nested outer-fold selected rates:',[f'{r:.3%}' for r in behavior_rate_by_fold])
print(f'Final evaluation rate (median outer selection): {FINAL_BEHAVIOR_POSITIVE_RATE:.3%}')
print(f'Actual weighted OOF mass selected: {selected_population_mass:.3%}')
print(f'Exact labeled behavior MAP: {behavior_map_labeled:.4f}')
print(f'PU weighted-stable behavior MAP surrogate: {behavior_map_stress:.4f}')
print(f'Positive-family accuracy: {family_accuracy:.4f}')
for name,score in zip(behavior_names,behavior_ap_labeled):
    print(f'exact labeled {name:<24} {score:.4f}')


Activation mode: risk
Nested outer-fold selected rates: ['1.000%', '1.500%', '2.000%', '1.000%', '2.000%']
Final evaluation rate (median outer selection): 1.500%
Actual weighted OOF mass selected: 1.509%
Exact labeled behavior MAP: 0.9124
PU weighted-stable behavior MAP surrogate: 0.5580
Positive-family accuracy: 0.9812
exact labeled directed_transfer        0.8982
exact labeled soft_play                0.9954
exact labeled coordinated_isolation    0.8435


## 12. Strict family-specific evidence rankers

The V2a stacked evidence model reused the global `oof_behavior` array inside each evidence-training fold. For an outer evidence fold, some training-row behavior probabilities came from classifiers that had seen the outer validation behavior labels.

V2.1 removes that dependency entirely:

- three rankers are trained separately for `directed_transfer`, `soft_play`, and `coordinated_isolation`;
- behavior probabilities are **not evidence-training features**;
- for each outer validation fold, the three ranker scores are normalized within pair and mixed only with that fold's clean outer-OOF behavior probabilities;
- the model/heuristic weight is fixed before OOF evaluation;
- the 25k deployment routing budget is simulated in OOF using the corresponding population fraction.


In [18]:
# ============================================================
# 12A. EVIDENCE TABLE — NO STACKED BEHAVIOR TRAINING FEATURES
# ============================================================

positive_info = (
    dev_labels.filter(pl.col('label') == 1)
    .select(['pair_id','behavior_family'])
    .with_columns(
        pl.when(pl.col('behavior_family') == 'directed_transfer').then(1)
        .when(pl.col('behavior_family') == 'soft_play').then(2)
        .otherwise(3).cast(pl.Int8).alias('behavior_id_true')
    )
)

evidence_keys = (
    dev_evidence.select(['pair_id','hand_id']).unique()
    .with_columns(pl.lit(1,dtype=pl.Int8).alias('is_evidence'))
)

pair_fold_context = pl.DataFrame({'pair_id':train_df['pair_id'],'fold':fold_id})

EVIDENCE_BASE_COLS = [
    'net_gap_bb','pair_contribution_bb','opposite_net_sign','true_hu_actions','pair_bets','pair_calls','pair_river_aggression',
    'pair_river_checks','true_hu_checks','true_hu_calls','any_pocket_pair','mean_hole_rank_sum','both_showdown','max_amount_pot_ratio',
    'immediate_outsider_folds_after_aggression','pair_raises','pair_overbets','outsider_folds_to_pair','max_amount_bb',
    'directed_score_v2','soft_score_v2','isolation_score_v2','suspicious_score_v2'
]

RANK_BASE = list(dict.fromkeys(
    SCORE_FEATURES + [c for c in EVIDENCE_BASE_COLS if c in HAND_FEATURES]
))[:28]
RANK_FEATURES = [f'{c}_pair_pct' for c in RANK_BASE]


def add_family_heuristics(lf):
    directed_h = (
        pl.col('net_gap_bb').log1p()
        + 0.60*pl.col('pair_contribution_bb').log1p()
        + 1.10*pl.col('opposite_net_sign')
        + 0.45*(pl.col('pair_bets')+pl.col('pair_calls')).log1p()
        + 0.40*pl.col('true_hu_actions').log1p()
        + 0.35*pl.col('pair_river_aggression').log1p()
    )
    soft_h = (
        1.35*pl.col('pair_river_checks').log1p()
        + 0.80*(pl.col('true_hu_checks')+pl.col('true_hu_calls')).log1p()
        + 0.55*pl.col('pair_calls').log1p()
        + 0.65*pl.col('any_pocket_pair')
        + 0.08*pl.col('mean_hole_rank_sum')
        - 0.10*pl.col('max_amount_pot_ratio')
    )
    iso_h = (
        1.80*pl.col('immediate_outsider_folds_after_aggression').log1p()
        + 0.75*pl.col('pair_raises').log1p()
        + 0.45*pl.col('pair_overbets').log1p()
        + 0.35*pl.col('outsider_folds_to_pair').log1p()
        + 0.20*pl.col('max_amount_bb').log1p()
    )
    return lf.with_columns(
        directed_h.cast(pl.Float32).alias('heuristic_directed'),
        soft_h.cast(pl.Float32).alias('heuristic_soft'),
        iso_h.cast(pl.Float32).alias('heuristic_isolation')
    )


evidence_base = (
    pl.scan_parquet(DEV_HAND_FEATURES_PATH)
    .select(['pair_id','hand_id',*HAND_FEATURES])
    .join(positive_info.lazy(),on='pair_id')
    .join(evidence_keys.lazy(),on=['pair_id','hand_id'],how='left')
    .with_columns(pl.col('is_evidence').fill_null(0))
    .collect()
    .join(pair_fold_context,on='pair_id')
    .with_columns([
        (pl.col(c).rank(method='average').over('pair_id') / pl.len().over('pair_id')).cast(pl.Float32).alias(f'{c}_pair_pct')
        for c in RANK_BASE
    ])
    .with_row_index('_row')
)
evidence_base = add_family_heuristics(evidence_base.lazy()).collect()

EVIDENCE_HEURISTIC_COLS = ['heuristic_directed','heuristic_soft','heuristic_isolation']
evidence_feature_cols = HAND_FEATURES + RANK_FEATURES + EVIDENCE_HEURISTIC_COLS

print(f'Evidence candidate hands: {len(evidence_base):,}')
print(f'Positive evidence rows:   {evidence_base["is_evidence"].sum():,}')
print(f'Evidence model features:  {len(evidence_feature_cols):,}')

source_evidence_key_count=dev_evidence_keys_unique.height
joined_evidence_key_count=(
    evidence_base.filter(pl.col('is_evidence') == 1)
    .select(['pair_id','hand_id']).unique().height
)
assert development_evidence_membership_validated is True
assert joined_evidence_key_count == source_evidence_key_count, (
    f'Evidence join recovered {joined_evidence_key_count} positive keys; source has {source_evidence_key_count}.'
)
assert int(evidence_base['is_evidence'].sum()) == source_evidence_key_count
assert evidence_base['pair_id'].n_unique() == positive_info['pair_id'].n_unique() == 372
assert not any(c.startswith('family_') for c in evidence_feature_cols), 'Behavior probabilities must not be evidence training features.'
evidence_training_membership_validated=True


Evidence candidate hands: 45,129
Positive evidence rows:   1,817
Evidence model features:  102


In [19]:
# ============================================================
# 12B. STRICT FAMILY-SPECIFIC OOF XGB RANKERS
# ============================================================


def evidence_matrix(df):
    d = df.sort(['pair_id','hand_id'])
    matrix = d.select(evidence_feature_cols).to_numpy().astype(np.float32)
    matrix = np.nan_to_num(matrix,nan=0,posinf=0,neginf=0)
    target = d['is_evidence'].to_numpy().astype(np.int8)
    groups = d.group_by('pair_id',maintain_order=True).len()['len'].to_numpy()
    return d,matrix,target,groups


def within_pair_percentile(pair_ids,scores):
    frame = pd.DataFrame({'pair_id':pair_ids,'score':scores})
    return frame.groupby('pair_id',sort=False)['score'].rank(method='average',pct=True).to_numpy(dtype=np.float32)


rank_base_params = dict(
    learning_rate=0.032,max_depth=4,min_child_weight=2,
    subsample=0.90,colsample_bytree=0.86,reg_alpha=0.12,reg_lambda=4.5,
    objective='rank:pairwise',eval_metric='map@5',tree_method='hist',n_jobs=-1
)

n_ev = len(evidence_base)
evidence_oof_family_raw = np.zeros((n_ev,3),dtype=np.float32)
evidence_oof_modelmix = np.zeros(n_ev,dtype=np.float32)
evidence_oof_heurmix = np.zeros(n_ev,dtype=np.float32)
family_best_rounds = {1:[],2:[],3:[]}
evidence_cv_manifest=[]

pair_oof_behavior = pd.DataFrame({
    'pair_id':pair_ids_np,
    'family_directed_prob':oof_behavior[:,0],
    'family_soft_prob':oof_behavior[:,1],
    'family_isolation_prob':oof_behavior[:,2]
}).set_index('pair_id')

for outer_fold in tqdm(range(N_FOLDS),desc='Strict evidence CV'):
    tr_df = evidence_base.filter(pl.col('fold') != outer_fold)
    va_df = evidence_base.filter(pl.col('fold') == outer_fold)

    # One OUTER-TRAIN fold is used only for ranker early stopping.
    candidate_stop_folds = [f for f in range(N_FOLDS) if f != outer_fold]
    stop_fold = candidate_stop_folds[0]

    va_ordered,Xva_all,yva,gva = evidence_matrix(va_df)
    row_ids = va_ordered['_row'].to_numpy()

    outer_train_pairs=set(tr_df['pair_id'].unique().to_list())
    outer_validation_pairs=set(va_df['pair_id'].unique().to_list())
    stop_pairs=set(tr_df.filter(pl.col('fold') == stop_fold)['pair_id'].unique().to_list())
    probe_train_pairs=outer_train_pairs-stop_pairs
    fold_manifest={
        'outer_fold':int(outer_fold),
        'stop_fold':int(stop_fold),
        'outer_train_validation_disjoint':bool(outer_train_pairs.isdisjoint(outer_validation_pairs)),
        'stop_excludes_outer_validation':bool(stop_pairs.isdisjoint(outer_validation_pairs)),
        'probe_train_stop_disjoint':bool(probe_train_pairs.isdisjoint(stop_pairs)),
        'outer_train_pair_n':int(len(outer_train_pairs)),
        'outer_validation_pair_n':int(len(outer_validation_pairs)),
        'stop_pair_n':int(len(stop_pairs)),
    }
    assert all([
        fold_manifest['outer_train_validation_disjoint'],
        fold_manifest['stop_excludes_outer_validation'],
        fold_manifest['probe_train_stop_disjoint'],
    ])
    evidence_cv_manifest.append(fold_manifest)

    for family_id in (1,2,3):
        fam_all_tr = tr_df.filter(pl.col('behavior_id_true') == family_id)
        fam_probe_train = fam_all_tr.filter(pl.col('fold') != stop_fold)
        fam_probe_stop = fam_all_tr.filter(pl.col('fold') == stop_fold)
        assert fam_probe_train['pair_id'].n_unique() > 0 and fam_probe_stop['pair_id'].n_unique() > 0

        _,Xprobe,yprobe,gprobe = evidence_matrix(fam_probe_train)
        _,Xstop,ystop,gstop = evidence_matrix(fam_probe_stop)

        probe = XGBRanker(
            **rank_base_params,n_estimators=1050,early_stopping_rounds=80,
            random_state=SEED+1000+100*outer_fold+family_id
        )
        probe.fit(
            Xprobe,yprobe,group=gprobe,
            eval_set=[(Xstop,ystop)],eval_group=[gstop],verbose=False
        )
        rounds = max(1,int(probe.best_iteration)+1)
        family_best_rounds[family_id].append(rounds)

        _,Xtr,ytr,gtr = evidence_matrix(fam_all_tr)
        model = XGBRanker(
            **rank_base_params,n_estimators=rounds,
            random_state=SEED+2000+100*outer_fold+family_id
        )
        model.fit(Xtr,ytr,group=gtr,verbose=False)
        evidence_oof_family_raw[row_ids,family_id-1] = model.predict(Xva_all)

    va_pair_ids = va_ordered['pair_id'].to_numpy()
    probs = pair_oof_behavior.loc[va_pair_ids][
        ['family_directed_prob','family_soft_prob','family_isolation_prob']
    ].to_numpy(dtype=np.float32)

    model_pct = np.column_stack([
        within_pair_percentile(va_pair_ids,evidence_oof_family_raw[row_ids,k])
        for k in range(3)
    ])
    heur_raw = va_ordered.select(EVIDENCE_HEURISTIC_COLS).to_numpy().astype(np.float32)
    heur_pct = np.column_stack([
        within_pair_percentile(va_pair_ids,heur_raw[:,k]) for k in range(3)
    ])

    evidence_oof_modelmix[row_ids] = np.sum(probs*model_pct,axis=1)
    evidence_oof_heurmix[row_ids] = np.sum(probs*heur_pct,axis=1)

    fold_fullblend = (
        EVIDENCE_MODEL_WEIGHT*evidence_oof_modelmix[row_ids]
        + (1-EVIDENCE_MODEL_WEIGHT)*evidence_oof_heurmix[row_ids]
    )
    print(
        f'Fold {outer_fold+1}: full-blend MAP@5='
        f'{evidence_map5(va_pair_ids,va_ordered["hand_id"].to_numpy(),yva,fold_fullblend):.4f}'
    )

print('Family ranker median rounds:',{
    behavior_names[k-1]:int(np.median(v)) for k,v in family_best_rounds.items()
})


Strict evidence CV:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 1: full-blend MAP@5=0.3927
Fold 2: full-blend MAP@5=0.3495
Fold 3: full-blend MAP@5=0.2996
Fold 4: full-blend MAP@5=0.3552
Fold 5: full-blend MAP@5=0.3670
Family ranker median rounds: {np.str_('directed_transfer'): 37, np.str_('soft_play'): 45, np.str_('coordinated_isolation'): 71}


In [20]:
# ============================================================
# 12C. ROUTED OOF EVIDENCE SCORE + FINAL FAMILY RANKERS
# ============================================================

pair_ids_ev = evidence_base['pair_id'].to_numpy()
hand_ids_ev = evidence_base['hand_id'].to_numpy()
y_ev = evidence_base['is_evidence'].to_numpy().astype(np.int8)

evidence_oof_fullblend = (
    EVIDENCE_MODEL_WEIGHT*evidence_oof_modelmix
    + (1-EVIDENCE_MODEL_WEIGHT)*evidence_oof_heurmix
).astype(np.float32)

# Match the production top-25k routing budget using the same fraction of estimated population mass.
EVIDENCE_MODEL_PAIR_RATE = EVIDENCE_MODEL_PAIR_LIMIT / len(eval_pairs_prep)
route_pair_mask = weighted_top_mask(oof_risk,population_weight,EVIDENCE_MODEL_PAIR_RATE)
route_lookup = dict(zip(pair_ids_np,route_pair_mask))
route_row_mask = np.array([route_lookup[p] for p in pair_ids_ev],dtype=bool)

evidence_oof_routed = np.where(
    route_row_mask,evidence_oof_fullblend,evidence_oof_heurmix
).astype(np.float32)

full_blend_map = evidence_map5(pair_ids_ev,hand_ids_ev,y_ev,evidence_oof_fullblend)
overall_evidence_map = evidence_map5(pair_ids_ev,hand_ids_ev,y_ev,evidence_oof_routed)
positive_pair_ids = evidence_base.select('pair_id').unique()['pair_id'].to_list()
positive_route_coverage = float(np.mean([route_lookup[p] for p in positive_pair_ids]))

print(f'Fixed evidence model weight: {EVIDENCE_MODEL_WEIGHT:.3f}')
print(f'Production model-route rate: {EVIDENCE_MODEL_PAIR_RATE:.3%}')
print(f'OOF positive-pair route coverage: {positive_route_coverage:.3%}')
print(f'OOF full-model-blend MAP@5 (diagnostic): {full_blend_map:.4f}')
print(f'OOF ROUTED Evidence MAP@5 (submission-matched): {overall_evidence_map:.4f}')
for behavior_id,name in enumerate(behavior_names,1):
    mask = evidence_base['behavior_id_true'].to_numpy() == behavior_id
    print(f'{name:<24} {evidence_map5(pair_ids_ev[mask],hand_ids_ev[mask],y_ev[mask],evidence_oof_routed[mask]):.4f}')

# Train final family rankers on all disclosed positive pairs with fixed rounds from strict outer CV.
final_evidence_models = {}
final_evidence_rounds = {}
for family_id,name in enumerate(behavior_names,1):
    fam_df = evidence_base.filter(pl.col('behavior_id_true') == family_id)
    _,Xfam,yfam,gfam = evidence_matrix(fam_df)
    rounds = int(np.median(family_best_rounds[family_id]))
    model = XGBRanker(
        **rank_base_params,n_estimators=rounds,
        random_state=SEED+3000+family_id
    )
    model.fit(Xfam,yfam,group=gfam,verbose=False)
    final_evidence_models[family_id] = model
    final_evidence_rounds[name] = rounds

print('Final family evidence rounds:',final_evidence_rounds)
y_evidence = y_ev


Fixed evidence model weight: 0.700
Production model-route rate: 22.214%
OOF positive-pair route coverage: 98.656%
OOF full-model-blend MAP@5 (diagnostic): 0.3529
OOF ROUTED Evidence MAP@5 (submission-matched): 0.3488
directed_transfer        0.3344
soft_play                0.4004
coordinated_isolation    0.2978
Final family evidence rounds: {np.str_('directed_transfer'): 37, np.str_('soft_play'): 45, np.str_('coordinated_isolation'): 71}


## 13. Exact nested OOF composite

This composite uses:

- strict outer-fold pair risk predictions;
- **fold-specific behavior activation rates selected only on inner routing-tune rows**;
- strict family-ranker evidence OOF predictions;
- the same 25k evidence-model routing fraction used by submission inference;
- the supplied stable row-wise AP implementation;
- explicit deterministic evidence tie-breaking by `hand_id`.

The behavior route is therefore no longer inherited from the development-tuned 5% rule.


In [21]:
# ============================================================
# 13. EXACT STRICT KNOWN-LABEL OOF COMPOSITE
# ============================================================

solution_known = dev_labels.to_pandas()[['pair_id','label','behavior_family']].rename(
    columns={'label':'risk_score','behavior_family':'predicted_behavior'}
)
for rank in range(1,6):
    m = (
        dev_evidence.filter(pl.col('evidence_rank') == rank)
        .select(['pair_id','hand_id']).to_pandas().set_index('pair_id')['hand_id']
    )
    solution_known[f'evidence_hand_{rank}'] = solution_known['pair_id'].map(m).fillna(NO_EVIDENCE)

# Routed OOF evidence top-5.
ev_pred = pd.DataFrame({
    'pair_id':pair_ids_ev,
    'hand_id':evidence_base['hand_id'].to_numpy(),
    'score':evidence_oof_routed
}).sort_values(['pair_id','score','hand_id'],ascending=[True,False,True],kind='mergesort')
ev_top = ev_pred.groupby('pair_id',sort=False).head(5).copy()
ev_top['rank'] = ev_top.groupby('pair_id',sort=False).cumcount()+1
ev_wide = ev_top.pivot(index='pair_id',columns='rank',values='hand_id')

known_idx = np.flatnonzero(known)
known_pair_ids = pair_ids_np[known_idx]
known_family = family_oof[known_idx]
known_active = behavior_active_oof[known_idx]
known_pred_behavior = np.where(known_active,behavior_names[known_family],'none')

submission_known = pd.DataFrame({
    'pair_id':known_pair_ids,
    'risk_score':oof_risk[known_idx],
    'predicted_behavior':known_pred_behavior
})
for rank in range(1,6):
    mapping = ev_wide[rank] if rank in ev_wide.columns else pd.Series(dtype=object)
    submission_known[f'evidence_hand_{rank}'] = submission_known['pair_id'].map(mapping).fillna(NO_EVIDENCE)

known_components = competition_components(solution_known,submission_known)
print(json.dumps(known_components,indent=2))
print('✅ Exact strict OOF competition-metric pipeline executed successfully.')


{
  "pair_ap": 0.9737033653053857,
  "evidence_map5": 0.34878658900836323,
  "behavior_map": 0.9123960784910484,
  "final_score": 0.8425892813645475,
  "behavior_class_ap": {
    "directed_transfer": 0.8982017687156835,
    "soft_play": 0.9954373577805742,
    "coordinated_isolation": 0.8435491089768874
  }
}
✅ Exact strict OOF competition-metric pipeline executed successfully.


## 14. Evaluation inference

CV ensembles are used for pair risk and behavior. The exact feature columns are asserted to match development before prediction.

In [22]:
# ============================================================
# 14. EVALUATION PAIR PREDICTIONS — SAME FIXED ROUTING RULES
# ============================================================

missing_eval_features = [c for c in feature_cols if c not in eval_features.columns]
assert not missing_eval_features, f'Evaluation matrix missing features: {missing_eval_features[:20]}'

X_eval = (
    eval_features.select(feature_cols).to_pandas()
    .replace([np.inf,-np.inf],np.nan)
    .fillna(0)
    .astype(np.float32)
)

eval_risk = np.mean([m.predict_proba(X_eval)[:,1] for m in risk_models],axis=0).astype(np.float32)
eval_behavior_proba = np.mean([m.predict_proba(X_eval) for m in behavior_models],axis=0).astype(np.float32)
eval_family = eval_behavior_proba.argmax(axis=1)
eval_conf = eval_behavior_proba.max(axis=1)

if BEHAVIOR_ACTIVATION_MODE == 'risk_x_confidence':
    eval_activation = eval_risk*np.clip(eval_conf,0.05,1.0)
else:
    eval_activation = eval_risk

behavior_active_eval = weighted_top_mask(
    eval_activation,np.ones(len(eval_activation),dtype=float),FINAL_BEHAVIOR_POSITIVE_RATE
)
predicted_behavior = np.where(behavior_active_eval,behavior_names[eval_family],'none')

eval_predictions = pl.DataFrame({
    'pair_id':eval_features['pair_id'],
    'risk_score':eval_risk,
    'predicted_behavior':predicted_behavior,
    'family_directed_prob':eval_behavior_proba[:,0],
    'family_soft_prob':eval_behavior_proba[:,1],
    'family_isolation_prob':eval_behavior_proba[:,2]
})

print(f'Pairs: {len(eval_predictions):,}')
print(f'Final nested-derived behavior rate: {FINAL_BEHAVIOR_POSITIVE_RATE:.3%}')
print(f'Predicted non-none behaviors: {behavior_active_eval.sum():,}')
display(eval_predictions.group_by('predicted_behavior').len().sort('len',descending=True))


Pairs: 112,540
Final nested-derived behavior rate: 1.500%
Predicted non-none behaviors: 1,689


predicted_behavior,len
str,u32
"""none""",110851
"""directed_transfer""",684
"""coordinated_isolation""",591
"""soft_play""",414


## H2 — Exact Hold'em semantics and counterfactual action residuals

This section is the only score-seeking change relative to the V2.2 source.
Behavior predictions remain frozen; H2 can modify pair ranking and evidence
ranking only if their strict development validation gates pass.


In [23]:
# ============================================================
# H2A. EXACT HOLDEM EVALUATOR + SELECTED PAIR-HAND SEMANTICS
# ============================================================

try:
    from numba import njit, prange
    H2_NUMBA_AVAILABLE = True
except Exception:
    H2_NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        def deco(fn):
            return fn
        return deco
    prange = range

print('H2 numba available:', H2_NUMBA_AVAILABLE)

H2_RANK_BASE = 15
H2_CAT_SCALE = H2_RANK_BASE ** 5
H2_MAX_KEY = 8 * H2_CAT_SCALE + sum(14 * (H2_RANK_BASE ** p) for p in range(5))


@njit()
def _h2_straight_high(mask):
    # Broadway through six-high.
    for hi in range(14, 5 - 1, -1):
        ok = True
        for r in range(hi, hi - 5, -1):
            if (mask & (1 << r)) == 0:
                ok = False
                break
        if ok:
            return hi
    # Wheel A-2-3-4-5.
    wheel = (1 << 14) | (1 << 5) | (1 << 4) | (1 << 3) | (1 << 2)
    if (mask & wheel) == wheel:
        return 5
    return 0


@njit()
def _h2_pack(cat, a=0, b=0, c=0, d=0, e=0):
    return (
        cat * (H2_RANK_BASE ** 5)
        + a * (H2_RANK_BASE ** 4)
        + b * (H2_RANK_BASE ** 3)
        + c * (H2_RANK_BASE ** 2)
        + d * H2_RANK_BASE
        + e
    )


@njit()
def _h2_eval7_row(cards):
    """Exact best-five-card ordering for 5-7 valid cards.

    Card encoding: (rank-2)*4 + suit, rank 2..14, suit 0..3.
    Missing cards are -1. When fewer than five total cards exist, a deterministic
    pre-showdown fallback key is returned so folded/preflop-only hands remain usable.
    """
    rank_count = [0] * 15
    suit_count = [0] * 4
    suit_mask = [0] * 4
    rank_mask = 0
    n = 0

    for code in cards:
        if code < 0:
            continue
        rank = code // 4 + 2
        suit = code % 4
        rank_count[rank] += 1
        suit_count[suit] += 1
        suit_mask[suit] |= (1 << rank)
        rank_mask |= (1 << rank)
        n += 1

    # Fewer than five cards: exact preflop/partial-board ordering is not a
    # five-card Hold'em hand, so use a deterministic semantic fallback.
    if n < 5:
        hi = 0
        lo = 0
        pair_rank = 0
        for r in range(14, 1, -1):
            if rank_count[r] >= 2 and pair_rank == 0:
                pair_rank = r
            if rank_count[r] > 0:
                if hi == 0:
                    hi = r
                elif lo == 0:
                    lo = r
        if pair_rank > 0:
            return _h2_pack(1, pair_rank, hi, lo, 0, 0)
        return _h2_pack(0, hi, lo, 0, 0, 0)

    # Straight flush.
    best_sf = 0
    for s in range(4):
        if suit_count[s] >= 5:
            sh = _h2_straight_high(suit_mask[s])
            if sh > best_sf:
                best_sf = sh
    if best_sf > 0:
        return _h2_pack(8, best_sf, 0, 0, 0, 0)

    # Four of a kind.
    quad = 0
    for r in range(14, 1, -1):
        if rank_count[r] == 4:
            quad = r
            break
    if quad > 0:
        kicker = 0
        for r in range(14, 1, -1):
            if r != quad and rank_count[r] > 0:
                kicker = r
                break
        return _h2_pack(7, quad, kicker, 0, 0, 0)

    # Full house.
    trip1 = 0
    trip2 = 0
    for r in range(14, 1, -1):
        if rank_count[r] >= 3:
            if trip1 == 0:
                trip1 = r
            elif trip2 == 0:
                trip2 = r
    if trip1 > 0:
        pair = trip2
        if pair == 0:
            for r in range(14, 1, -1):
                if r != trip1 and rank_count[r] >= 2:
                    pair = r
                    break
        if pair > 0:
            return _h2_pack(6, trip1, pair, 0, 0, 0)

    # Flush.
    best_flush_key = -1
    for s in range(4):
        if suit_count[s] >= 5:
            vals = [0, 0, 0, 0, 0]
            k = 0
            for r in range(14, 1, -1):
                if (suit_mask[s] & (1 << r)) != 0:
                    vals[k] = r
                    k += 1
                    if k == 5:
                        break
            key = _h2_pack(5, vals[0], vals[1], vals[2], vals[3], vals[4])
            if key > best_flush_key:
                best_flush_key = key
    if best_flush_key >= 0:
        return best_flush_key

    # Straight.
    sh = _h2_straight_high(rank_mask)
    if sh > 0:
        return _h2_pack(4, sh, 0, 0, 0, 0)

    # Trips.
    if trip1 > 0:
        kick = [0, 0]
        k = 0
        for r in range(14, 1, -1):
            if r != trip1 and rank_count[r] > 0:
                kick[k] = r
                k += 1
                if k == 2:
                    break
        return _h2_pack(3, trip1, kick[0], kick[1], 0, 0)

    # Two pair / one pair.
    pairs = [0, 0, 0]
    npairs = 0
    for r in range(14, 1, -1):
        if rank_count[r] >= 2:
            if npairs < 3:
                pairs[npairs] = r
            npairs += 1

    if npairs >= 2:
        p1 = pairs[0]
        p2 = pairs[1]
        kicker = 0
        for r in range(14, 1, -1):
            if r != p1 and r != p2 and rank_count[r] > 0:
                kicker = r
                break
        return _h2_pack(2, p1, p2, kicker, 0, 0)

    if npairs == 1:
        p = pairs[0]
        kick = [0, 0, 0]
        k = 0
        for r in range(14, 1, -1):
            if r != p and rank_count[r] > 0:
                kick[k] = r
                k += 1
                if k == 3:
                    break
        return _h2_pack(1, p, kick[0], kick[1], kick[2], 0)

    # High card.
    high = [0, 0, 0, 0, 0]
    k = 0
    for r in range(14, 1, -1):
        if rank_count[r] > 0:
            high[k] = r
            k += 1
            if k == 5:
                break
    return _h2_pack(0, high[0], high[1], high[2], high[3], high[4])


@njit(parallel=True)
def _h2_eval_batch(cards):
    out = np.empty(cards.shape[0], dtype=np.int64)
    for i in prange(cards.shape[0]):
        out[i] = _h2_eval7_row(cards[i])
    return out


H2_CARD_RANKS = '23456789TJQKA'
H2_CARD_SUITS = 'cdhs'
H2_CARD_TO_CODE = {
    f'{r}{s}': (ri * 4 + si)
    for ri, r in enumerate(H2_CARD_RANKS)
    for si, s in enumerate(H2_CARD_SUITS)
}


def _h2_card_code_expr(column):
    txt = pl.col(column).cast(pl.Utf8)
    rank = txt.str.replace(r'[cdhsCDHS]$', '').str.to_uppercase()
    suit = txt.str.slice(-1, 1).str.to_lowercase()
    rank_num = (
        pl.when(rank == 'A').then(14)
        .when(rank == 'K').then(13)
        .when(rank == 'Q').then(12)
        .when(rank == 'J').then(11)
        .when(rank == 'T').then(10)
        .otherwise(rank.cast(pl.Int16, strict=False))
    )
    suit_num = (
        pl.when(suit == 'c').then(0)
        .when(suit == 'd').then(1)
        .when(suit == 'h').then(2)
        .when(suit == 's').then(3)
        .otherwise(-1)
    )
    return (
        pl.when(rank_num.is_null() | (rank_num < 2) | (suit_num < 0))
        .then(-1)
        .otherwise((rank_num - 2) * 4 + suit_num)
        .cast(pl.Int16)
    )


# Evaluator self-test before touching the competition rows.
def _h2_codes(cards):
    arr = np.full((1, 7), -1, dtype=np.int16)
    for j, c in enumerate(cards[:7]):
        arr[0, j] = H2_CARD_TO_CODE[c]
    return arr

_h2_test_hands = [
    (['As','Ks','Qs','Js','Ts'], 8),   # royal/straight flush
    (['As','Ah','Ad','Ac','Kd'], 7),   # quads
    (['As','Ah','Ad','Kc','Kd'], 6),   # full house
    (['As','Js','9s','5s','2s'], 5),   # flush
    (['As','2h','3d','4c','5s'], 4),   # wheel straight
    (['As','Ah','Ad','Kc','Qd'], 3),   # trips
    (['As','Ah','Kd','Kc','Qd'], 2),   # two pair
    (['As','Ah','Kd','Qc','Jd'], 1),   # pair
    (['As','Kd','Qc','9c','7d'], 0),   # high card
]
_h2_test_categories = []
for cards, expected in _h2_test_hands:
    key = int(_h2_eval_batch(_h2_codes(cards))[0])
    cat = int(key // H2_CAT_SCALE)
    _h2_test_categories.append(cat)
    assert cat == expected, (cards, cat, expected)

assert _h2_test_categories == list(range(8, -1, -1))
h2_exact_evaluator_selftest_passed = True
print('✅ H2 exact evaluator self-test:', _h2_test_categories)


H2_SOURCE_COLS = list(H2_REQUIRED_HAND_SOURCE_COLS)

H2_HAND_FEATURES = [
    'h2_category_1','h2_category_2','h2_strength_1','h2_strength_2',
    'h2_strength_max','h2_strength_min','h2_strength_gap','h2_strength_mean',
    'h2_aggr_resid_1','h2_aggr_resid_2','h2_passive_resid_1','h2_passive_resid_2',
    'h2_fold_resid_1','h2_fold_resid_2',
    'h2_strong_passivity_hu','h2_strength_weighted_hu_passive',
    'h2_transfer_strength_gap','h2_transfer_strong_donor','h2_transfer_weak_donor',
    'h2_aggr_outsider_residual',
    'h2_directed_semantic','h2_soft_semantic','h2_isolation_semantic','h2_generic_semantic'
]
H2_TAIL_FEATURES = [
    'h2_strong_passivity_hu','h2_strength_weighted_hu_passive',
    'h2_transfer_strength_gap','h2_transfer_strong_donor','h2_transfer_weak_donor',
    'h2_aggr_outsider_residual',
    'h2_directed_semantic','h2_soft_semantic','h2_isolation_semantic','h2_generic_semantic'
]


def build_h2_semantic_hands(hand_features_path, pair_ids, phase, out_path):
    if out_path.exists():
        return pl.read_parquet(out_path)

    schema = pl.scan_parquet(hand_features_path).collect_schema().names()
    missing = [c for c in H2_SOURCE_COLS if c not in schema]
    assert not missing, f'H2 source hand features missing: {missing}'

    seat_base = pl.scan_parquet(SEATS_PATH).select(
        ['hand_id','player_id','hole_card_1','hole_card_2']
    )
    s1 = seat_base.rename({
        'player_id':'player_1',
        'hole_card_1':'_h11',
        'hole_card_2':'_h12'
    })
    s2 = seat_base.rename({
        'player_id':'player_2',
        'hole_card_1':'_h21',
        'hole_card_2':'_h22'
    })

    bcols = ['base_fold_rate','base_check_rate','base_call_rate','base_aggression_rate']
    base = (
        pl.scan_parquet(PLAYER_ACTION_BASELINE_PATH)
        .filter(pl.col('phase') == phase)
        .select(['player_id', *bcols])
    )
    b1 = base.rename({'player_id':'player_1', **{c:f'{c}_1' for c in bcols}})
    b2 = base.rename({'player_id':'player_2', **{c:f'{c}_2' for c in bcols}})

    pair_lookup = dev_pairs_prep if phase == 'development' else eval_pairs_prep
    pair_lookup_lf = pair_lookup.lazy().select(['pair_id','player_1','player_2'])

    lf = (
        pl.scan_parquet(hand_features_path)
        .filter(pl.col('pair_id').is_in(pair_ids))
        .select(H2_SOURCE_COLS)
        .join(pair_lookup_lf,on='pair_id',how='left')
        .join(s1,on=['hand_id','player_1'],how='left')
        .join(s2,on=['hand_id','player_2'],how='left')
        .join(
            pl.scan_parquet(HANDS_PATH).select(['hand_id','board_cards']),
            on='hand_id',how='left'
        )
        .join(b1,on='player_1',how='left')
        .join(b2,on='player_2',how='left')
        .with_columns([
            pl.col('board_cards').fill_null('').str.split(' ').list.get(i, null_on_oob=True).alias(f'_b{i+1}')
            for i in range(5)
        ])
        .with_columns([
            _h2_card_code_expr(c).alias(f'_code_{c}')
            for c in ['_h11','_h12','_h21','_h22','_b1','_b2','_b3','_b4','_b5']
        ])
    )

    df = lf.collect(engine='streaming')
    print(f'H2 {phase} semantic candidate hand rows: {len(df):,}')
    assert len(df) > 0
    assert df['player_1'].null_count() == 0 and df['player_2'].null_count() == 0
    assert df.select(['pair_id','hand_id']).unique().height == len(df), 'Duplicate H2 pair-hand rows after semantic joins.'

    board_cols = [f'_code__b{i}' for i in range(1,6)]
    cards1 = np.column_stack([
        df['_code__h11'].to_numpy(),
        df['_code__h12'].to_numpy(),
        *[df[c].to_numpy() for c in board_cols],
    ]).astype(np.int16, copy=False)
    cards2 = np.column_stack([
        df['_code__h21'].to_numpy(),
        df['_code__h22'].to_numpy(),
        *[df[c].to_numpy() for c in board_cols],
    ]).astype(np.int16, copy=False)

    key1 = _h2_eval_batch(cards1)
    key2 = _h2_eval_batch(cards2)
    del cards1, cards2
    assert np.all((key1 >= 0) & (key1 <= H2_MAX_KEY))
    assert np.all((key2 >= 0) & (key2 <= H2_MAX_KEY))

    cat1 = (key1 // H2_CAT_SCALE).astype(np.int8)
    cat2 = (key2 // H2_CAT_SCALE).astype(np.int8)
    str1 = (key1.astype(np.float64) / float(H2_MAX_KEY)).astype(np.float32)
    str2 = (key2.astype(np.float64) / float(H2_MAX_KEY)).astype(np.float32)

    df = df.with_columns([
        pl.Series('h2_category_1',cat1),
        pl.Series('h2_category_2',cat2),
        pl.Series('h2_strength_1',str1),
        pl.Series('h2_strength_2',str2),
    ])

    den1 = pl.max_horizontal(pl.col('n_actions_1').cast(pl.Float32),pl.lit(1.0))
    den2 = pl.max_horizontal(pl.col('n_actions_2').cast(pl.Float32),pl.lit(1.0))
    pass1 = (pl.col('checks_1') + pl.col('calls_1')).cast(pl.Float32) / den1
    pass2 = (pl.col('checks_2') + pl.col('calls_2')).cast(pl.Float32) / den2
    aggr1 = pl.col('aggressive_actions_1').cast(pl.Float32) / den1
    aggr2 = pl.col('aggressive_actions_2').cast(pl.Float32) / den2
    fold1 = pl.col('folds_1').cast(pl.Float32) / den1
    fold2 = pl.col('folds_2').cast(pl.Float32) / den2

    df = df.with_columns(
        pl.max_horizontal('h2_strength_1','h2_strength_2').cast(pl.Float32).alias('h2_strength_max'),
        pl.min_horizontal('h2_strength_1','h2_strength_2').cast(pl.Float32).alias('h2_strength_min'),
        (pl.col('h2_strength_1')-pl.col('h2_strength_2')).abs().cast(pl.Float32).alias('h2_strength_gap'),
        ((pl.col('h2_strength_1')+pl.col('h2_strength_2'))/2).cast(pl.Float32).alias('h2_strength_mean'),
        (aggr1-pl.col('base_aggression_rate_1').fill_null(0)).cast(pl.Float32).alias('h2_aggr_resid_1'),
        (aggr2-pl.col('base_aggression_rate_2').fill_null(0)).cast(pl.Float32).alias('h2_aggr_resid_2'),
        (pass1-(pl.col('base_check_rate_1').fill_null(0)+pl.col('base_call_rate_1').fill_null(0))).cast(pl.Float32).alias('h2_passive_resid_1'),
        (pass2-(pl.col('base_check_rate_2').fill_null(0)+pl.col('base_call_rate_2').fill_null(0))).cast(pl.Float32).alias('h2_passive_resid_2'),
        (fold1-pl.col('base_fold_rate_1').fill_null(0)).cast(pl.Float32).alias('h2_fold_resid_1'),
        (fold2-pl.col('base_fold_rate_2').fill_null(0)).cast(pl.Float32).alias('h2_fold_resid_2'),
    )

    df = df.with_columns(
        (
            pl.max_horizontal(
                pl.col('h2_strength_1')*pl.col('h2_passive_resid_1').clip(lower_bound=0),
                pl.col('h2_strength_2')*pl.col('h2_passive_resid_2').clip(lower_bound=0)
            )
            * pl.col('true_hu_actions').cast(pl.Float32).log1p()
        ).cast(pl.Float32).alias('h2_strong_passivity_hu'),
        (
            (
                pl.col('h2_strength_1')*pl.col('p1_hu_passive').cast(pl.Float32)
                + pl.col('h2_strength_2')*pl.col('p2_hu_passive').cast(pl.Float32)
            )
            / (pl.col('true_hu_actions').cast(pl.Float32)+1.0)
        ).cast(pl.Float32).alias('h2_strength_weighted_hu_passive'),
        (
            pl.col('transfer_any_bb').cast(pl.Float32).log1p()
            * pl.col('h2_strength_gap')
        ).cast(pl.Float32).alias('h2_transfer_strength_gap'),
        (
            pl.col('transfer_1_to_2_bb')*(pl.col('h2_strength_1')-pl.col('h2_strength_2')).clip(lower_bound=0)
            + pl.col('transfer_2_to_1_bb')*(pl.col('h2_strength_2')-pl.col('h2_strength_1')).clip(lower_bound=0)
        ).cast(pl.Float32).alias('h2_transfer_strong_donor'),
        (
            pl.col('transfer_1_to_2_bb')*(pl.col('h2_strength_2')-pl.col('h2_strength_1')).clip(lower_bound=0)
            + pl.col('transfer_2_to_1_bb')*(pl.col('h2_strength_1')-pl.col('h2_strength_2')).clip(lower_bound=0)
        ).cast(pl.Float32).alias('h2_transfer_weak_donor'),
        (
            pl.max_horizontal(
                pl.col('h2_aggr_resid_1').clip(lower_bound=0),
                pl.col('h2_aggr_resid_2').clip(lower_bound=0)
            )
            * pl.col('immediate_outsider_folds_after_aggression').cast(pl.Float32).log1p()
        ).cast(pl.Float32).alias('h2_aggr_outsider_residual'),
    )

    df = df.with_columns(
        (
            1.00*pl.col('h2_transfer_strength_gap').log1p()
            + 0.60*pl.col('h2_transfer_strong_donor').log1p()
            + 0.45*pl.col('h2_transfer_weak_donor').log1p()
            + 0.75*pl.col('opposite_net_sign').cast(pl.Float32)
            + 0.25*pl.col('pair_contribution_bb').cast(pl.Float32).log1p()
        ).cast(pl.Float32).alias('h2_directed_semantic'),
        (
            1.20*pl.col('h2_strong_passivity_hu')
            + 0.90*pl.col('h2_strength_weighted_hu_passive')
            + 0.35*(pl.col('pair_river_checks')+pl.col('true_hu_checks')+pl.col('true_hu_calls')).cast(pl.Float32).log1p()
            - 0.20*pl.col('pair_river_aggression').cast(pl.Float32).log1p()
        ).cast(pl.Float32).alias('h2_soft_semantic'),
        (
            1.25*pl.col('h2_aggr_outsider_residual')
            + 0.70*pl.col('immediate_outsider_folds_after_aggression').cast(pl.Float32).log1p()
            + 0.30*pl.col('postflop_outsider_folds_after_aggression').cast(pl.Float32).log1p()
            + 0.20*pl.col('pair_raises').cast(pl.Float32).log1p()
        ).cast(pl.Float32).alias('h2_isolation_semantic'),
    ).with_columns(
        pl.max_horizontal(
            'h2_directed_semantic','h2_soft_semantic','h2_isolation_semantic'
        ).cast(pl.Float32).alias('h2_generic_semantic')
    )

    keep = ['pair_id','hand_id',*H2_HAND_FEATURES]
    df.select(keep).write_parquet(out_path,compression='zstd')
    return df.select(keep)


def aggregate_h2_pairs(path):
    lf = pl.scan_parquet(path)
    agg = []
    agg += [pl.col(c).mean().alias(f'{c}_mean') for c in H2_HAND_FEATURES]
    agg += [pl.col(c).max().alias(f'{c}_max') for c in H2_HAND_FEATURES]
    agg += [pl.col(c).quantile(0.95, interpolation='nearest').alias(f'{c}_p95') for c in H2_TAIL_FEATURES]
    agg += [pl.col(c).top_k(3).mean().alias(f'{c}_top3') for c in H2_TAIL_FEATURES]
    agg += [pl.col(c).top_k(5).mean().alias(f'{c}_top5') for c in H2_TAIL_FEATURES]
    return lf.group_by('pair_id').agg(agg).collect(engine='streaming')


# Development H2 semantics first. Evaluation semantics are intentionally deferred
# until the strict H2 validation gate passes, avoiding unnecessary work on a failed experiment.
h2_dev_pair_ids = (
    dev_labels.select('pair_id').unique()['pair_id'].to_list()
)
h2_dev_hands = build_h2_semantic_hands(
    DEV_HAND_FEATURES_PATH,h2_dev_pair_ids,'development',H2_DEV_SEMANTIC_HAND_PATH
)
h2_dev_pairs = aggregate_h2_pairs(H2_DEV_SEMANTIC_HAND_PATH)
assert h2_dev_pairs['pair_id'].n_unique() == dev_labels['pair_id'].n_unique()

eval_predictions = eval_predictions.with_columns(
    pl.col('risk_score').alias('base_risk_score')
)
h2_base_predicted_behavior = eval_predictions['predicted_behavior'].to_list()

print('H2 development pair features:',h2_dev_pairs.shape)


# ============================================================
# H2B. SECOND-STAGE PAIR MODEL — BEHAVIOR MODEL REMAINS FROZEN
# ============================================================

known_pair_context = pl.DataFrame({
    'pair_id':np.asarray(train_df['pair_id'].to_list(),dtype=object)[known],
    'label':y[known].astype(np.int8),
    'fold':fold_id[known].astype(np.int8),
    'base_oof_risk':oof_risk[known].astype(np.float32),
}).sort('pair_id')

h2_pair_train = h2_dev_pairs.join(known_pair_context,on='pair_id',how='inner').sort('pair_id')
assert len(h2_pair_train) == len(dev_labels), 'H2 must cover every disclosed labeled pair.'

# Stacking hygiene: base_oof_risk is NOT a training feature. It is only used in
# the fixed external blend after the semantic model produces strict OOF scores.
H2_PAIR_FEATURE_COLS = [
    c for c,dtype in h2_pair_train.schema.items()
    if c not in {'pair_id','label','fold','base_oof_risk'} and dtype.is_numeric()
]
assert H2_PAIR_FEATURE_COLS
assert 'base_oof_risk' not in H2_PAIR_FEATURE_COLS

Xh2 = (
    h2_pair_train.select(H2_PAIR_FEATURE_COLS).to_pandas()
    .replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
)
yh2 = h2_pair_train['label'].to_numpy().astype(np.int8)
h2_pair_fold = h2_pair_train['fold'].to_numpy().astype(np.int8)
base_h2 = h2_pair_train['base_oof_risk'].to_numpy().astype(np.float32)

h2_pair_oof = np.zeros(len(h2_pair_train),dtype=np.float32)
h2_pair_seen = np.zeros(len(h2_pair_train),dtype=bool)

for fold in range(N_FOLDS):
    tr = np.flatnonzero(h2_pair_fold != fold)
    va = np.flatnonzero(h2_pair_fold == fold)
    assert len(tr) and len(va)
    assert not h2_pair_seen[va].any()

    model = XGBClassifier(
        n_estimators=H2_PAIR_N_ESTIMATORS,
        learning_rate=0.035,
        max_depth=3,
        min_child_weight=5,
        subsample=0.90,
        colsample_bytree=0.80,
        reg_alpha=0.15,
        reg_lambda=4.0,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1,
        random_state=SEED+2200+fold,
    )
    model.fit(Xh2.iloc[tr],yh2[tr],verbose=False)
    h2_pair_oof[va] = model.predict_proba(Xh2.iloc[va])[:,1]
    h2_pair_seen[va] = True

assert h2_pair_seen.all()
assert np.isfinite(h2_pair_oof).all()

def _h2_rank_pct(x):
    return pd.Series(np.asarray(x,dtype=float)).rank(method='average',pct=True).to_numpy(dtype=np.float64)

h2_base_pct = _h2_rank_pct(base_h2)
h2_meta_pct = _h2_rank_pct(h2_pair_oof)
h2_pair_blend_oof = (
    (1.0-H2_PAIR_BLEND_WEIGHT)*h2_base_pct
    + H2_PAIR_BLEND_WEIGHT*h2_meta_pct
)

h2_base_pair_ap = _average_precision_exact(yh2,base_h2)
h2_pair_ap = _average_precision_exact(yh2,h2_pair_blend_oof)
h2_pair_gain = h2_pair_ap-h2_base_pair_ap
H2_PAIR_PRELIMINARY_ENABLED = bool(h2_pair_gain >= H2_MIN_PAIR_AP_GAIN)
H2_PAIR_ENABLED = H2_PAIR_PRELIMINARY_ENABLED

print(f'H2 pair base exact AP: {h2_base_pair_ap:.6f}')
print(f'H2 pair blend exact AP: {h2_pair_ap:.6f} | gain={h2_pair_gain:+.6f}')
print('H2 pair preliminary gate:',H2_PAIR_PRELIMINARY_ENABLED)


# ============================================================
# H2C. FAMILY-SPECIFIC EXACT-SEMANTIC EVIDENCE RANKERS
# ============================================================

h2_positive_info = (
    dev_labels.filter(pl.col('label') == 1)
    .select(['pair_id','behavior_family'])
    .with_columns(
        pl.when(pl.col('behavior_family') == 'directed_transfer').then(1)
        .when(pl.col('behavior_family') == 'soft_play').then(2)
        .otherwise(3).cast(pl.Int8).alias('behavior_id_true')
    )
)

h2_base_evidence_frame = pl.DataFrame({
    'pair_id':pair_ids_ev,
    'hand_id':hand_ids_ev,
    '_v22_evidence_score':evidence_oof_routed,
})
assert h2_base_evidence_frame.select(['pair_id','hand_id']).unique().height == len(h2_base_evidence_frame)

h2_ev = (
    pl.scan_parquet(H2_DEV_SEMANTIC_HAND_PATH)
    .join(h2_positive_info.lazy(),on='pair_id')
    .join(evidence_keys.lazy(),on=['pair_id','hand_id'],how='left')
    .with_columns(pl.col('is_evidence').fill_null(0))
    .collect()
    .join(pair_fold_context,on='pair_id')
    .join(h2_base_evidence_frame,on=['pair_id','hand_id'],how='left')
)

assert len(h2_ev) == len(evidence_base), 'H2/base evidence candidate row counts differ.'
assert h2_ev['_v22_evidence_score'].null_count() == 0, 'H2/base evidence key alignment failed.'
h2_evidence_key_alignment_validated = True

H2_RANK_BASE_COLS = [
    'h2_strength_gap','h2_strong_passivity_hu','h2_transfer_strength_gap',
    'h2_transfer_strong_donor','h2_transfer_weak_donor',
    'h2_aggr_outsider_residual',
    'h2_directed_semantic','h2_soft_semantic','h2_isolation_semantic','h2_generic_semantic'
]
H2_RANK_FEATURES = [f'{c}_pair_pct' for c in H2_RANK_BASE_COLS]

h2_ev = h2_ev.with_columns([
    (pl.col(c).rank(method='average').over('pair_id') / pl.len().over('pair_id'))
    .cast(pl.Float32).alias(f'{c}_pair_pct')
    for c in H2_RANK_BASE_COLS
])

H2_EVIDENCE_FEATURE_COLS = H2_HAND_FEATURES + H2_RANK_FEATURES


def h2_evidence_matrix(df):
    d = df.sort(['pair_id','hand_id'])
    X = d.select(H2_EVIDENCE_FEATURE_COLS).to_numpy().astype(np.float32)
    X = np.nan_to_num(X,nan=0,posinf=0,neginf=0)
    y = d['is_evidence'].to_numpy().astype(np.float32)
    groups = d.group_by('pair_id',maintain_order=True).len()['len'].to_numpy().astype(np.int32)
    return d,X,y,groups


h2_ev_oof_family_raw = np.zeros((len(h2_ev),3),dtype=np.float32)
h2_ev_seen = np.zeros(len(h2_ev),dtype=bool)
h2_ev = h2_ev.with_row_index('_h2_row')
h2_final_evidence_models = {}

for outer_fold in range(N_FOLDS):
    va = h2_ev.filter(pl.col('fold') == outer_fold)
    va_ordered,Xva_all,yva_all,gva_all = h2_evidence_matrix(va)
    va_ids = va_ordered['_h2_row'].to_numpy()
    assert not h2_ev_seen[va_ids].any(), 'H2 evidence OOF row predicted more than once.'
    h2_ev_seen[va_ids] = True

    for family_id in (1,2,3):
        tr = h2_ev.filter(
            (pl.col('fold') != outer_fold) &
            (pl.col('behavior_id_true') == family_id)
        )
        _,Xtr,ytr,gtr = h2_evidence_matrix(tr)

        model = XGBRanker(
            n_estimators=H2_EVIDENCE_N_ESTIMATORS,
            learning_rate=0.035,
            max_depth=3,
            min_child_weight=2,
            subsample=0.90,
            colsample_bytree=0.85,
            reg_alpha=0.10,
            reg_lambda=4.0,
            objective='rank:pairwise',
            eval_metric='map@5',
            tree_method='hist',
            n_jobs=-1,
            random_state=SEED+3300+100*outer_fold+family_id,
        )
        model.fit(Xtr,ytr,group=gtr,verbose=False)

        # Strict inference simulation: each family specialist scores every
        # positive pair in the untouched outer validation fold.
        h2_ev_oof_family_raw[va_ids,family_id-1] = model.predict(Xva_all).astype(np.float32)

assert h2_ev_seen.all()
assert np.isfinite(h2_ev_oof_family_raw).all()

h2_ev_pair_ids = h2_ev['pair_id'].to_numpy()
h2_ev_hand_ids = h2_ev['hand_id'].to_numpy()
h2_ev_y = h2_ev['is_evidence'].to_numpy().astype(np.int8)

h2_model_pct = np.column_stack([
    within_pair_percentile(h2_ev_pair_ids,h2_ev_oof_family_raw[:,k])
    for k in range(3)
])

h2_probs = pair_oof_behavior.loc[h2_ev_pair_ids][
    ['family_directed_prob','family_soft_prob','family_isolation_prob']
].to_numpy(dtype=np.float32)

h2_evidence_mix = np.sum(h2_probs*h2_model_pct,axis=1)
h2_base_evidence_scores = h2_ev['_v22_evidence_score'].to_numpy().astype(np.float32)
h2_evidence_blend_oof = (
    (1.0-H2_EVIDENCE_BLEND_WEIGHT)*h2_base_evidence_scores
    + H2_EVIDENCE_BLEND_WEIGHT*h2_evidence_mix
)

h2_base_evidence_map = evidence_map5(
    h2_ev_pair_ids,h2_ev_hand_ids,h2_ev_y,h2_base_evidence_scores
)
h2_evidence_map = evidence_map5(
    h2_ev_pair_ids,h2_ev_hand_ids,h2_ev_y,h2_evidence_blend_oof
)
h2_evidence_gain = h2_evidence_map-h2_base_evidence_map

h2_family_maps = {}
h2_family_base_maps = {}
for family_id,name in enumerate(behavior_names,start=1):
    mask = h2_ev['behavior_id_true'].to_numpy() == family_id
    h2_family_base_maps[name] = evidence_map5(
        h2_ev_pair_ids[mask],h2_ev_hand_ids[mask],h2_ev_y[mask],h2_base_evidence_scores[mask]
    )
    h2_family_maps[name] = evidence_map5(
        h2_ev_pair_ids[mask],h2_ev_hand_ids[mask],h2_ev_y[mask],h2_evidence_blend_oof[mask]
    )

h2_family_nonworse = sum(
    h2_family_maps[n] + 0.005 >= h2_family_base_maps[n]
    for n in behavior_names
)
H2_EVIDENCE_PRELIMINARY_ENABLED = bool(
    h2_evidence_gain >= H2_MIN_EVIDENCE_MAP_GAIN
    and h2_family_nonworse >= 2
)
H2_EVIDENCE_ENABLED = H2_EVIDENCE_PRELIMINARY_ENABLED

print(f'H2 evidence base MAP@5: {h2_base_evidence_map:.6f}')
print(f'H2 evidence blend MAP@5: {h2_evidence_map:.6f} | gain={h2_evidence_gain:+.6f}')
print('H2 evidence family base:',h2_family_base_maps)
print('H2 evidence family blend:',h2_family_maps)
print('H2 evidence preliminary gate:',H2_EVIDENCE_PRELIMINARY_ENABLED)

# ============================================================
# H2D. STRICT COMBINED OOF COMPETITION-METRIC GATE
# ============================================================

h2_submission_known = submission_known.copy()

if H2_PAIR_ENABLED:
    h2_pair_score_map = dict(zip(
        h2_pair_train['pair_id'].to_list(),
        h2_pair_blend_oof.astype(float)
    ))
    h2_submission_known['risk_score'] = (
        h2_submission_known['pair_id'].map(h2_pair_score_map)
        .fillna(h2_submission_known['risk_score'])
        .astype(float)
    )

if H2_EVIDENCE_ENABLED:
    h2_ev_pred = pd.DataFrame({
        'pair_id':h2_ev_pair_ids,
        'hand_id':h2_ev_hand_ids,
        'score':h2_evidence_blend_oof,
    }).sort_values(['pair_id','score','hand_id'],ascending=[True,False,True],kind='mergesort')
    h2_ev_top = h2_ev_pred.groupby('pair_id',sort=False).head(5).copy()
    h2_ev_top['rank'] = h2_ev_top.groupby('pair_id',sort=False).cumcount()+1
    h2_ev_wide = h2_ev_top.pivot(index='pair_id',columns='rank',values='hand_id')
    for rank in range(1,6):
        mapping = h2_ev_wide[rank] if rank in h2_ev_wide.columns else pd.Series(dtype=object)
        h2_submission_known[f'evidence_hand_{rank}'] = (
            h2_submission_known['pair_id'].map(mapping)
            .fillna(h2_submission_known[f'evidence_hand_{rank}'])
        )

h2_known_components = competition_components(solution_known,h2_submission_known)
h2_composite_gain = float(h2_known_components['final_score'] - known_components['final_score'])
H2_KEEP_CANDIDATE = bool(
    (H2_PAIR_ENABLED or H2_EVIDENCE_ENABLED)
    and h2_composite_gain >= H2_MIN_COMPOSITE_GAIN
)

print('H2 combined strict known-label components:')
print(json.dumps(h2_known_components,indent=2))
print(f'H2 composite gain vs same-run V2.2: {h2_composite_gain:+.6f}')
print('H2 final keep candidate:',H2_KEEP_CANDIDATE)

if not H2_KEEP_CANDIDATE:
    H2_PAIR_ENABLED = False
    H2_EVIDENCE_ENABLED = False

# Only now build evaluation semantics. Failed H2 runs stop here without the
# multi-million-row evaluation semantic join/evaluator cost.
h2_eval_pair_ids = []
h2_eval_pairs = pl.DataFrame({'pair_id':[]},schema={'pair_id':pl.Utf8})
h2_final_pair_model = None
h2_final_evidence_models = {}

if H2_KEEP_CANDIDATE:
    h2_eval_pair_ids = (
        eval_predictions
        .sort(['base_risk_score','pair_id'],descending=[True,False])
        .head(min(H2_EVAL_PAIR_LIMIT,len(eval_predictions)))['pair_id'].to_list()
    )
    h2_eval_hands = build_h2_semantic_hands(
        EVAL_HAND_FEATURES_PATH,h2_eval_pair_ids,'evaluation',H2_EVAL_SEMANTIC_HAND_PATH
    )
    h2_eval_pairs = aggregate_h2_pairs(H2_EVAL_SEMANTIC_HAND_PATH)
    assert h2_eval_pairs['pair_id'].n_unique() == len(h2_eval_pair_ids)
    print('H2 evaluation candidate pair features:',h2_eval_pairs.shape)

    if H2_PAIR_ENABLED:
        h2_eval_pair_matrix = h2_eval_pairs.sort('pair_id')
        missing_pair_features = [c for c in H2_PAIR_FEATURE_COLS if c not in h2_eval_pair_matrix.columns]
        assert not missing_pair_features,missing_pair_features
        Xh2_eval = (
            h2_eval_pair_matrix.select(H2_PAIR_FEATURE_COLS).to_pandas()
            .replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
        )

        h2_final_pair_model = XGBClassifier(
            n_estimators=H2_PAIR_N_ESTIMATORS,
            learning_rate=0.035,max_depth=3,min_child_weight=5,
            subsample=0.90,colsample_bytree=0.80,
            reg_alpha=0.15,reg_lambda=4.0,
            objective='binary:logistic',eval_metric='logloss',
            tree_method='hist',n_jobs=-1,random_state=SEED+2299
        )
        h2_final_pair_model.fit(Xh2,yh2,verbose=False)
        h2_eval_prob = h2_final_pair_model.predict_proba(Xh2_eval)[:,1].astype(np.float32)
        h2_eval_pair_matrix = h2_eval_pair_matrix.with_columns(
            pl.Series('_h2_pair_prob',h2_eval_prob)
        )

        base_all = eval_predictions['base_risk_score'].to_numpy().astype(np.float64)
        base_all_pct = _h2_rank_pct(base_all)
        cand_h2_pct = _h2_rank_pct(h2_eval_pair_matrix['_h2_pair_prob'].to_numpy().astype(np.float64))

        cutoff = max(0.0,1.0-len(h2_eval_pair_matrix)/len(eval_predictions))
        mapped_h2_pct = cutoff + (1.0-cutoff)*cand_h2_pct

        pair_to_row = {p:i for i,p in enumerate(eval_predictions['pair_id'].to_list())}
        final_pct = base_all_pct.copy()
        for p,h2p in zip(h2_eval_pair_matrix['pair_id'].to_list(),mapped_h2_pct):
            i = pair_to_row[p]
            final_pct[i] = (
                (1.0-H2_PAIR_BLEND_WEIGHT)*base_all_pct[i]
                + H2_PAIR_BLEND_WEIGHT*h2p
            )
        eval_predictions = eval_predictions.with_columns(
            pl.Series('risk_score',final_pct.astype(np.float32))
        )

    if H2_EVIDENCE_ENABLED:
        for family_id in (1,2,3):
            fam = h2_ev.filter(pl.col('behavior_id_true') == family_id)
            _,Xf,yf,gf = h2_evidence_matrix(fam)
            model = XGBRanker(
                n_estimators=H2_EVIDENCE_N_ESTIMATORS,
                learning_rate=0.035,max_depth=3,min_child_weight=2,
                subsample=0.90,colsample_bytree=0.85,
                reg_alpha=0.10,reg_lambda=4.0,
                objective='rank:pairwise',eval_metric='map@5',
                tree_method='hist',n_jobs=-1,
                random_state=SEED+3399+family_id
            )
            model.fit(Xf,yf,group=gf,verbose=False)
            h2_final_evidence_models[family_id] = model

if not H2_PAIR_ENABLED:
    eval_predictions = eval_predictions.with_columns(
        pl.col('base_risk_score').alias('risk_score')
    )

assert eval_predictions['predicted_behavior'].to_list() == h2_base_predicted_behavior
h2_behavior_pipeline_frozen = True

print('✅ H2 semantic pair/evidence layer complete.')



H2 numba available: True
✅ H2 exact evaluator self-test: [8, 7, 6, 5, 4, 3, 2, 1, 0]
H2 development semantic candidate hand rows: 223,749
H2 development pair features: (1860, 79)
H2 pair base exact AP: 0.973703
H2 pair blend exact AP: 0.971047 | gain=-0.002656
H2 pair preliminary gate: False
H2 evidence base MAP@5: 0.348787
H2 evidence blend MAP@5: 0.391369 | gain=+0.042583
H2 evidence family base: {np.str_('directed_transfer'): 0.3344350600600601, np.str_('soft_play'): 0.4003956228956229, np.str_('coordinated_isolation'): 0.29782608695652174}
H2 evidence family blend: {np.str_('directed_transfer'): 0.4375675675675676, np.str_('soft_play'): 0.4206523569023569, np.str_('coordinated_isolation'): 0.27503623188405796}
H2 evidence preliminary gate: True
H2 combined strict known-label components:
{
  "pair_ap": 0.9737033653053857,
  "evidence_map5": 0.39136947431302277,
  "behavior_map": 0.9123960784910484,
  "final_score": 0.8511058584254794,
  "behavior_class_ap": {
    "directed_transfer"

## 15. Evidence inference

Every evaluation pair gets a heuristic top-5. For the highest-risk pairs, the trained ranker re-scores all shared evaluation hands and the OOF-selected model/heuristic blend replaces the heuristic list.

This preserves evidence coverage even when the pair ranker under-ranks a hidden positive.

In [24]:
# ============================================================
# 15A. HEURISTIC TOP-5 FOR ALL EVALUATION PAIRS
#      SAME FAMILY-PROBABILITY MIXTURE AS STRICT OOF
# ============================================================

behavior_context_eval = eval_predictions.select([
    'pair_id','family_directed_prob','family_soft_prob','family_isolation_prob'
])

heuristic_needed = list(dict.fromkeys(EVIDENCE_BASE_COLS))
heuristic_lf = (
    pl.scan_parquet(EVAL_HAND_FEATURES_PATH)
    .select(['pair_id','hand_id',*heuristic_needed])
)
heuristic_lf = add_family_heuristics(heuristic_lf)
heuristic_lf = (
    heuristic_lf
    .with_columns([
        (pl.col(c).rank(method='average').over('pair_id') / pl.len().over('pair_id')).cast(pl.Float32).alias(f'_{c}_pct')
        for c in EVIDENCE_HEURISTIC_COLS
    ])
    .join(behavior_context_eval.lazy(),on='pair_id')
    .with_columns(
        (
            pl.col('family_directed_prob')*pl.col('_heuristic_directed_pct')
            + pl.col('family_soft_prob')*pl.col('_heuristic_soft_pct')
            + pl.col('family_isolation_prob')*pl.col('_heuristic_isolation_pct')
        ).cast(pl.Float32).alias('heuristic_mixture')
    )
)

heuristic_top = (
    heuristic_lf.group_by('pair_id').agg(
        pl.col('hand_id')
        .sort_by(['heuristic_mixture','hand_id'],descending=[True,False])
        .head(5)
        .alias('heuristic_hands')
    ).collect(engine='streaming')
)

print('Heuristic evidence coverage:',len(heuristic_top),'pairs')
assert len(heuristic_top) == len(eval_pairs_prep), 'Not every evaluation pair received heuristic evidence candidates.'


Heuristic evidence coverage: 112540 pairs


In [25]:
# ============================================================
# 15B. THREE FAMILY RANKERS FOR TOP-RISK PAIRS + FIXED BLEND
# ============================================================

model_limit = min(EVIDENCE_MODEL_PAIR_LIMIT,len(eval_predictions))
top_pair_ids = (
    eval_predictions.sort('risk_score',descending=True)
    .head(model_limit)['pair_id'].to_list()
)

model_hands = (
    pl.scan_parquet(EVAL_HAND_FEATURES_PATH)
    .filter(pl.col('pair_id').is_in(top_pair_ids))
    .select(['pair_id','hand_id',*HAND_FEATURES])
    .with_columns([
        (pl.col(c).rank(method='average').over('pair_id') / pl.len().over('pair_id')).cast(pl.Float32).alias(f'{c}_pair_pct')
        for c in RANK_BASE
    ])
)
model_hands = add_family_heuristics(model_hands).collect(engine='streaming')
print(f'Model evidence candidate rows: {len(model_hands):,}')

family_raw = np.empty((len(model_hands),3),dtype=np.float32)
for family_id in (1,2,3):
    model = final_evidence_models[family_id]
    for start in tqdm(
        range(0,len(model_hands),EVIDENCE_BATCH_SIZE),
        desc=f'Evidence family {behavior_names[family_id-1]}'
    ):
        end = min(start+EVIDENCE_BATCH_SIZE,len(model_hands))
        mat = model_hands.slice(start,end-start).select(evidence_feature_cols).to_numpy().astype(np.float32)
        mat = np.nan_to_num(mat,nan=0,posinf=0,neginf=0)
        family_raw[start:end,family_id-1] = model.predict(mat)

model_scored = model_hands.select(['pair_id','hand_id',*EVIDENCE_HEURISTIC_COLS])
for k,name in enumerate(['directed','soft','isolation']):
    model_scored = model_scored.with_columns(pl.Series(f'_model_{name}_raw',family_raw[:,k]))

model_scored = (
    model_scored
    .with_columns([
        (pl.col(f'_model_{name}_raw').rank(method='average').over('pair_id') / pl.len().over('pair_id')).cast(pl.Float32).alias(f'_model_{name}_pct')
        for name in ['directed','soft','isolation']
    ] + [
        (pl.col(c).rank(method='average').over('pair_id') / pl.len().over('pair_id')).cast(pl.Float32).alias(f'_{c}_pct')
        for c in EVIDENCE_HEURISTIC_COLS
    ])
    .join(behavior_context_eval,on='pair_id')
    .with_columns(
        (
            pl.col('family_directed_prob')*pl.col('_model_directed_pct')
            + pl.col('family_soft_prob')*pl.col('_model_soft_pct')
            + pl.col('family_isolation_prob')*pl.col('_model_isolation_pct')
        ).alias('_model_mixture'),
        (
            pl.col('family_directed_prob')*pl.col('_heuristic_directed_pct')
            + pl.col('family_soft_prob')*pl.col('_heuristic_soft_pct')
            + pl.col('family_isolation_prob')*pl.col('_heuristic_isolation_pct')
        ).alias('_heuristic_mixture')
    )
    .with_columns(
        (
            EVIDENCE_MODEL_WEIGHT*pl.col('_model_mixture')
            + (1-EVIDENCE_MODEL_WEIGHT)*pl.col('_heuristic_mixture')
        ).cast(pl.Float32).alias('_final_evidence_score')
    )
)

model_top = model_scored.group_by('pair_id').agg(
    pl.col('hand_id')
    .sort_by(['_final_evidence_score','hand_id'],descending=[True,False])
    .head(5)
    .alias('model_hands')
)
print('Model-ranked evidence pairs:',len(model_top))


# ============================================================
# H2E. OPTIONAL EXACT-SEMANTIC EVIDENCE BLEND
# Reassign model_top only when strict H2 evidence validation passed.
# ============================================================

if H2_EVIDENCE_ENABLED:
    h2_eval_model = (
        pl.scan_parquet(H2_EVAL_SEMANTIC_HAND_PATH)
        .filter(pl.col('pair_id').is_in(top_pair_ids))
        .collect(engine='streaming')
    )

    h2_missing_pairs = set(model_scored['pair_id'].unique().to_list()) - set(h2_eval_model['pair_id'].unique().to_list())
    print('H2 evidence semantic-cache missing model pairs:',len(h2_missing_pairs))
    assert len(h2_final_evidence_models) == 3
    assert len(h2_missing_pairs) == 0, 'H2 semantic cache does not cover every routed evidence pair.'

    h2_eval_model = h2_eval_model.with_columns([
        (pl.col(c).rank(method='average').over('pair_id') / pl.len().over('pair_id'))
        .cast(pl.Float32).alias(f'{c}_pair_pct')
        for c in H2_RANK_BASE_COLS
    ])

    model_scored = model_scored.join(
        h2_eval_model.select(['pair_id','hand_id',*H2_EVIDENCE_FEATURE_COLS]),
        on=['pair_id','hand_id'],how='left'
    )

    # Rows outside the H2 cache retain the frozen V2.2 evidence score.
    h2_raw_eval = np.zeros((len(model_scored),3),dtype=np.float32)
    h2_valid_mask = np.ones(len(model_scored),dtype=bool)
    h2_mat = model_scored.select(H2_EVIDENCE_FEATURE_COLS).to_numpy().astype(np.float32)
    h2_valid_mask = np.isfinite(h2_mat).all(axis=1)
    h2_mat = np.nan_to_num(h2_mat,nan=0,posinf=0,neginf=0)

    for family_id in (1,2,3):
        h2_model = h2_final_evidence_models[family_id]
        for start in range(0,len(model_scored),EVIDENCE_BATCH_SIZE):
            end = min(start+EVIDENCE_BATCH_SIZE,len(model_scored))
            h2_raw_eval[start:end,family_id-1] = h2_model.predict(h2_mat[start:end]).astype(np.float32)

    h2_pct_eval = np.column_stack([
        within_pair_percentile(model_scored['pair_id'].to_numpy(),h2_raw_eval[:,k])
        for k in range(3)
    ])
    h2_prob_eval = model_scored.select([
        'family_directed_prob','family_soft_prob','family_isolation_prob'
    ]).to_numpy().astype(np.float32)
    h2_mix_eval = np.sum(h2_prob_eval*h2_pct_eval,axis=1).astype(np.float32)

    base_evidence_eval = model_scored['_final_evidence_score'].to_numpy().astype(np.float32)
    final_h2_evidence_eval = base_evidence_eval.copy()
    final_h2_evidence_eval[h2_valid_mask] = (
        (1.0-H2_EVIDENCE_BLEND_WEIGHT)*base_evidence_eval[h2_valid_mask]
        + H2_EVIDENCE_BLEND_WEIGHT*h2_mix_eval[h2_valid_mask]
    )
    model_scored = model_scored.with_columns(
        pl.Series('_final_evidence_score',final_h2_evidence_eval)
    )

    model_top = model_scored.group_by('pair_id').agg(
        pl.col('hand_id')
        .sort_by(['_final_evidence_score','hand_id'],descending=[True,False])
        .head(5)
        .alias('model_hands')
    )
    print('✅ H2 exact-semantic evidence blend applied to cached model pairs.')
else:
    print('H2 evidence gate failed; V2.2 evidence ranking retained.')


Model evidence candidate rows: 2,077,695


Evidence family directed_transfer:   0%|          | 0/9 [00:00<?, ?it/s]

Evidence family soft_play:   0%|          | 0/9 [00:00<?, ?it/s]

Evidence family coordinated_isolation:   0%|          | 0/9 [00:00<?, ?it/s]

Model-ranked evidence pairs: 25000
H2 evidence semantic-cache missing model pairs: 0
✅ H2 exact-semantic evidence blend applied to cached model pairs.


In [26]:
# ============================================================
# 15C. MERGE MODEL / HEURISTIC EVIDENCE
# ============================================================

evidence_wide = (
    heuristic_top.join(model_top,on='pair_id',how='left')
    .with_columns(pl.coalesce(['model_hands','heuristic_hands']).alias('hands'))
    .with_columns([
        pl.col('hands').list.get(i,null_on_oob=True).fill_null(NO_EVIDENCE).alias(f'evidence_hand_{i+1}')
        for i in range(5)
    ])
    .select(['pair_id',*[f'evidence_hand_{i}' for i in range(1,6)]])
)

display(evidence_wide.head())

pair_id,evidence_hand_1,evidence_hand_2,evidence_hand_3,evidence_hand_4,evidence_hand_5
str,str,str,str,str,str
"""P1CC9740AD259""","""HF7683BAC3D382E""","""H2C17F0DC9A12DE""","""H6314EAB6663C66""","""H08BC63CEDE2146""","""H22B35F74141981"""
"""P6DD7AFB99C11""","""H5E5393EBE3E6A2""","""HB8ABBD397C5643""","""HA51534C1109BF5""","""H2D6E9680FC59ED""","""H313F25F3964314"""
"""PAFBD8573E4FF""","""HA4A9886EA0C231""","""H2008C097CB103E""","""H75605901A823EA""","""HE25EA1D2D0FE4E""","""H20FCC8146B8FEF"""
"""P07ED121A6DA2""","""HE64A603E9F0533""","""HC43864F4D72E8B""","""HB36FAD643687C4""","""H3F9F314B0D99F6""","""H14E4FF53BDFBAA"""
"""P750E22DDC661""","""H48FC72AA3A6013""","""HEAD14FA1ED2617""","""H2A7D5CC2A0F83A""","""H774DF91DD249FE""","""HE09F35B41D4645"""


## 16. Build and validate `submission.csv`

Validation mirrors the supplied metric / getting-started source constraints:

- exact pair coverage;
- unique pair IDs;
- risk in [0,1];
- allowed behavior strings;
- no null cells;
- no repeated evidence hand within a row;
- every submitted evidence hand is actually a shared **evaluation-phase** hand for that pair.

In [27]:
# ============================================================
# 16. SUBMISSION + STRICT VALIDATION
# ============================================================

submission = (
    sample_sub.select('pair_id')
    .join(
        eval_predictions.select(['pair_id','risk_score','predicted_behavior']),
        on='pair_id',how='left'
    )
    .join(evidence_wide,on='pair_id',how='left')
    .with_columns([
        pl.col(f'evidence_hand_{i}').fill_null(NO_EVIDENCE)
        for i in range(1,6)
    ])
    .select(sample_sub.columns)
)


def validate_submission_strict(submission_df, eval_pairs_df, valid_pair_hands_lf):
    sub_pd=submission_df.to_pandas()
    assert len(sub_pd) == len(eval_pairs_df), 'Submission row count mismatch.'
    assert sub_pd['pair_id'].is_unique, 'Duplicate pair_id in submission.'
    assert set(sub_pd['pair_id']) == set(eval_pairs_df['pair_id'].to_list()), 'pair_id coverage mismatch.'
    assert sub_pd['risk_score'].between(0,1).all(), 'risk_score outside [0,1].'
    assert set(sub_pd['predicted_behavior']).issubset(ALLOWED_BEHAVIORS), 'Invalid predicted_behavior.'
    assert not sub_pd.isna().any().any(), 'Submission contains null cells.'

    for row in sub_pd[list(EVIDENCE_COLUMNS)].itertuples(index=False,name=None):
        cleaned=_clean_evidence(list(row))
        assert len(cleaned) == len(set(cleaned)), 'Repeated evidence hand within one pair.'

    submitted_long=[]
    for rank in range(1,6):
        submitted_long.append(
            submission_df.select(['pair_id',pl.col(f'evidence_hand_{rank}').alias('hand_id')])
            .filter(pl.col('hand_id') != NO_EVIDENCE)
        )
    submitted_long=pl.concat(submitted_long).unique()
    invalid=(
        submitted_long.lazy()
        .join(valid_pair_hands_lf,on=['pair_id','hand_id'],how='anti')
        .collect(engine='streaming')
    )
    assert invalid.height == 0, f'{invalid.height} submitted evidence hands do not belong to the evaluation pair.'

    return True,{
        'rows':int(len(sub_pd)),
        'unique_pairs':int(sub_pd['pair_id'].nunique()),
        'submitted_evidence_keys':int(submitted_long.height),
        'invalid_evidence_keys':int(invalid.height),
    }


valid_pair_hands=pl.scan_parquet(EVAL_PAIR_HANDS_PATH).select(['pair_id','hand_id']).unique()
submission_validation_passed,submission_validation_manifest=validate_submission_strict(
    submission,eval_pairs,valid_pair_hands
)

SUBMISSION_PATH=Path('/kaggle/working/submission.csv')
submission.write_csv(SUBMISSION_PATH)
print(f'✅ Saved {SUBMISSION_PATH} | rows={len(submission):,}')
print('Submission validation manifest:',submission_validation_manifest)
display(submission.head())


✅ Saved /kaggle/working/submission.csv | rows=112,540
Submission validation manifest: {'rows': 112540, 'unique_pairs': 112540, 'submitted_evidence_keys': 562700, 'invalid_evidence_keys': 0}


pair_id,risk_score,predicted_behavior,evidence_hand_1,evidence_hand_2,evidence_hand_3,evidence_hand_4,evidence_hand_5
str,f32,str,str,str,str,str,str
"""P00005AC2A509""",0.001616,"""none""","""H06F2754F0AFCD8""","""H452A814BDCD3CA""","""H392FCB82F16534""","""H08BA1A1AEC0930""","""H758C7F184D9391"""
"""P00055C6C3588""",0.006103,"""none""","""H2BEE87002CFDD6""","""H05F67A02B804AF""","""H5C3252BD5DA395""","""HF60E526CDBECF7""","""H6352A933E8D333"""
"""P0007B4930526""",0.002243,"""none""","""H01137CB56D0044""","""H5A07605C51818F""","""H25200831CD91A0""","""HEEFC4AABC3C942""","""HD9CBC706DC9EE9"""
"""P000841AC274F""",0.003542,"""none""","""H218C00FE701F61""","""H40BCE1717DDB6C""","""H5983BF461E8818""","""HF01E88B2B0B272""","""H975439D2227113"""
"""P0008F9A8AD1F""",0.002533,"""none""","""HD47A3D82E64087""","""H2E04D935771C18""","""H9A3D5BAE02BD52""","""H566545DAF41804""","""H5BE4C4D9FD5848"""


## 17. Final runtime-derived verification gate

V2.2 no longer marks design claims as `True` inside the asserted gate. The asserted checks below are derived from runtime objects, fold manifests, evidence membership anti-joins, selected population mass, and the completed submission validator.

Descriptive design choices are reported separately from the runtime assertions.


In [28]:
# ============================================================
# 17. FINAL RUNTIME-DERIVED VERIFICATION GATE
# ============================================================

required_hand_v2={
    'opposite_net_sign','pair_river_checks','immediate_outsider_folds_after_aggression',
    'directed_score_v2','soft_score_v2','isolation_score_v2','suspicious_score_v2'
}
hand_cols=set(pl.scan_parquet(DEV_HAND_FEATURES_PATH).collect_schema().names())

required_pair_v2={
    'delta_check_mean','delta_call_mean','delta_raise_mean','delta_aggression_mean','delta_bet_to_pot_mean',
    'pool_pct_net_gap','pool_pct_combined_contrib','pool_pct_transfer_asymmetry',
    'net_direction_consistency','transfer_asymmetry',
    'directed_score_v2_top3','soft_score_v2_top3','isolation_score_v2_top3',
    'directed_score_v2_bin10_maxmean','soft_score_v2_bin10_maxmean','isolation_score_v2_bin10_maxmean',
    'exposure_log1p','exposure_confidence_20','directed_score_v2_rate95_smoothed',
    'soft_score_v2_rate95_smoothed','isolation_score_v2_rate95_smoothed'
}
missing_pair=required_pair_v2-set(dev_features.columns)

# Weighted-top selection may overshoot the requested mass by at most one selected row.
behavior_quota_checks=[]
for m in pair_cv_manifest:
    lower=m['outer_selected_mass'] + 1e-12 >= m['selected_behavior_rate']
    upper=m['outer_selected_mass'] <= m['selected_behavior_rate'] + m['outer_max_single_row_mass'] + 1e-12
    behavior_quota_checks.append(bool(lower and upper))

# Deterministic tie self-test: hand H1 must beat H2 on an exact score tie because H1 < H2.
tie_map_test=evidence_map5(
    np.array(['P','P'],dtype=object),
    np.array(['H2','H1'],dtype=object),
    np.array([0,1],dtype=np.int8),
    np.array([0.5,0.5],dtype=float)
)

runtime_checks={
    'source_schema_checked':bool(source_schema_checked),
    'metric_weights_70_20_10':bool(
        np.isclose(PAIR_METRIC_WEIGHT,0.70) and
        np.isclose(EVIDENCE_METRIC_WEIGHT,0.20) and
        np.isclose(BEHAVIOR_METRIC_WEIGHT,0.10) and
        np.isclose(PAIR_METRIC_WEIGHT+EVIDENCE_METRIC_WEIGHT+BEHAVIOR_METRIC_WEIGHT,1.0)
    ),
    'development_evidence_membership_validated':bool(
        development_evidence_membership_validated and evidence_training_membership_validated
    ),
    'pair_outer_train_validation_disjoint':all(m['outer_train_validation_disjoint'] for m in pair_cv_manifest),
    'pair_inner_fit_stop_tune_disjoint':all(m['inner_fit_stop_tune_disjoint'] for m in pair_cv_manifest),
    'pair_inner_subsets_inside_outer_train':all(m['inner_subsets_inside_outer_train'] for m in pair_cv_manifest),
    'pair_outer_validation_not_used_for_risk_early_stop':all(m['risk_stop_excludes_outer_validation'] for m in pair_cv_manifest),
    'pair_outer_validation_not_used_for_behavior_early_stop':all(m['behavior_stop_excludes_outer_validation'] for m in pair_cv_manifest),
    'pair_outer_validation_not_used_for_behavior_rate_tuning':all(m['routing_tune_excludes_outer_validation'] for m in pair_cv_manifest),
    'fold_specific_positive_player_weight_sources_outer_train_only':all(m['positive_weight_source_outer_train_only'] for m in pair_cv_manifest),
    'behavior_rate_nested_selected_for_all_folds':bool(
        len(behavior_rate_by_fold) == N_FOLDS and
        all(float(r) in set(BEHAVIOR_RATE_GRID.tolist()) for r in behavior_rate_by_fold)
    ),
    'behavior_population_weighted_quota_respected':all(behavior_quota_checks),
    'official_exact_ap_used_for_known_pair_reporting':bool(np.isfinite(primary_exact_pair_ap)),
    'evidence_behavior_probs_not_training_features':not any(c.startswith('family_') for c in evidence_feature_cols),
    'family_specific_evidence_rankers':len(final_evidence_models) == 3,
    'evidence_outer_train_validation_disjoint':all(m['outer_train_validation_disjoint'] for m in evidence_cv_manifest),
    'evidence_stop_excludes_outer_validation':all(m['stop_excludes_outer_validation'] for m in evidence_cv_manifest),
    'evidence_probe_train_stop_disjoint':all(m['probe_train_stop_disjoint'] for m in evidence_cv_manifest),
    'evidence_routing_simulated_oof':bool(np.isfinite(overall_evidence_map) and 0 <= positive_route_coverage <= 1),
    'evidence_deterministic_tie_break':bool(np.isclose(tie_map_test,1.0)),
    'counterparty_baselines_added':'delta_check_mean' in dev_features.columns,
    'pool_relative_features_added':'pool_pct_net_gap' in dev_features.columns,
    'episodic_features_added':'directed_score_v2_bin10_maxmean' in dev_features.columns,
    'isolation_sequence_motif_added':'immediate_outsider_folds_after_aggression' in hand_cols,
    'evidence_binary_relevance':set(np.unique(y_evidence)).issubset({0,1}),
    'strict_cache_rebuild_enabled':RESET_CACHE is True,
    'required_hand_features_present':required_hand_v2.issubset(hand_cols),
    'required_pair_features_present':not missing_pair,
    'submission_validated':bool(submission_validation_passed),
    'submission_file_exists':SUBMISSION_PATH.exists(),
    'known_composite_finite':bool(np.isfinite(known_components['final_score'])),
    'h2_early_preflight':bool(h2_early_preflight_passed),
    'h2_exact_evaluator_selftest':bool(h2_exact_evaluator_selftest_passed),
    'h2_dev_semantic_cache_exists':H2_DEV_SEMANTIC_HAND_PATH.exists(),
    'h2_eval_semantic_cache_condition':bool(
        (not H2_KEEP_CANDIDATE) or H2_EVAL_SEMANTIC_HAND_PATH.exists()
    ),
    'h2_pair_validation_finite':bool(np.isfinite(h2_pair_ap) and np.isfinite(h2_pair_gain)),
    'h2_evidence_validation_finite':bool(np.isfinite(h2_evidence_map) and np.isfinite(h2_evidence_gain)),
    'h2_evidence_key_alignment':bool(h2_evidence_key_alignment_validated),
    'h2_combined_metric_finite':bool(np.isfinite(h2_known_components['final_score']) and np.isfinite(h2_composite_gain)),
    'h2_behavior_pipeline_frozen':bool(h2_behavior_pipeline_frozen),
}

for k,v in runtime_checks.items():
    print(('✅' if v else '❌'),k)
assert all(runtime_checks.values()), 'Final runtime-derived verification gate failed.'

# Structural choices are reported, not asserted by manually setting booleans.
design_notes={
    'unknown_sampling':'constructed before positive-player lookup; sampled by table only',
    'behavior_rate_tuning_objective':BEHAVIOR_RATE_TUNING_OBJECTIVE,
    'final_behavior_rate_rule':'median of nested outer-fold selected rates',
    'evidence_model_weight_fixed':float(EVIDENCE_MODEL_WEIGHT),
    'evidence_model_pair_limit':int(EVIDENCE_MODEL_PAIR_LIMIT),
    'evidence_tie_break':'score descending, hand_id ascending',
    'cache_policy':'full derived-cache rebuild on Run All',
    'h2_exact_holdem':'exact best-five ordering from hole cards + completed board',
    'h2_counterfactual':'per-hand action residuals against each player baseline, strength-weighted',
    'h2_behavior_model_changed':False,
    'h2_eval_pair_limit':int(H2_EVAL_PAIR_LIMIT),
    'h2_pair_blend_weight':float(H2_PAIR_BLEND_WEIGHT),
    'h2_evidence_blend_weight':float(H2_EVIDENCE_BLEND_WEIGHT),
    'h2_min_composite_gain':float(H2_MIN_COMPOSITE_GAIN),
    'h2_pair_stacking_hygiene':'semantic-only H2 model; base OOF risk used only in fixed external blend',
    'h2_evidence_alignment':'V2.2 evidence OOF joined by (pair_id, hand_id), never positional',
    'h2_eval_semantics_policy':'built only after final H2 validation gate passes',
}

summary={
    'pipeline_build_id':PIPELINE_BUILD_ID,
    'cache_version':CACHE_VERSION,
    'n_pair_features':len(feature_cols),
    'strict_exact_labeled_pair_ap':float(primary_exact_pair_ap),
    'pu_weighted_stable_pair_ap_surrogate':float(primary_pu_surrogate),
    'behavior_rate_grid':list(map(float,BEHAVIOR_RATE_GRID)),
    'behavior_rate_by_outer_fold':list(map(float,behavior_rate_by_fold)),
    'behavior_rate_tune_score_by_outer_fold':list(map(float,behavior_rate_tune_score_by_fold)),
    'behavior_final_rate_median':float(FINAL_BEHAVIOR_POSITIVE_RATE),
    'behavior_activation_mode':BEHAVIOR_ACTIVATION_MODE,
    'exact_labeled_behavior_map':float(behavior_map_labeled),
    'evidence_oof_routed_map5':float(overall_evidence_map),
    'evidence_oof_fullblend_map5_diagnostic':float(full_blend_map),
    'evidence_model_weight_fixed':float(EVIDENCE_MODEL_WEIGHT),
    'evidence_model_pair_limit':int(EVIDENCE_MODEL_PAIR_LIMIT),
    'oof_positive_route_coverage':float(positive_route_coverage),
    'risk_best_rounds':list(map(int,risk_best_rounds)),
    'behavior_best_rounds':list(map(int,behavior_best_rounds)),
    'evidence_final_rounds':final_evidence_rounds,
    'pair_cv_manifest':pair_cv_manifest,
    'evidence_cv_manifest':evidence_cv_manifest,
    'submission_validation_manifest':submission_validation_manifest,
    'runtime_checks':runtime_checks,
    'design_notes':design_notes,
    'temporal_diagnostic':temporal_summary,
    'known_exact_components':known_components,
    'h2':{
        'public_lb_anchor_v22':float(V22_PUBLIC_LB_ANCHOR),
        'v22_local_anchor':{
            'pair_ap':float(V22_LOCAL_PAIR_AP_ANCHOR),
            'behavior_map':float(V22_LOCAL_BEHAVIOR_MAP_ANCHOR),
            'evidence_map5':float(V22_LOCAL_EVIDENCE_MAP_ANCHOR),
            'composite':float(V22_LOCAL_COMPOSITE_ANCHOR),
        },
        'numba_available':bool(H2_NUMBA_AVAILABLE),
        'pair_preliminary_enabled':bool(H2_PAIR_PRELIMINARY_ENABLED),
        'pair_enabled':bool(H2_PAIR_ENABLED),
        'pair_base_ap':float(h2_base_pair_ap),
        'pair_blend_ap':float(h2_pair_ap),
        'pair_gain':float(h2_pair_gain),
        'pair_min_gain_gate':float(H2_MIN_PAIR_AP_GAIN),
        'evidence_preliminary_enabled':bool(H2_EVIDENCE_PRELIMINARY_ENABLED),
        'evidence_enabled':bool(H2_EVIDENCE_ENABLED),
        'evidence_base_map5':float(h2_base_evidence_map),
        'evidence_blend_map5':float(h2_evidence_map),
        'evidence_gain':float(h2_evidence_gain),
        'evidence_min_gain_gate':float(H2_MIN_EVIDENCE_MAP_GAIN),
        'evidence_family_base':{k:float(v) for k,v in h2_family_base_maps.items()},
        'evidence_family_blend':{k:float(v) for k,v in h2_family_maps.items()},
        'evidence_family_nonworse':int(h2_family_nonworse),
        'eval_semantic_pair_count':int(len(h2_eval_pairs)),
        'combined_known_components':h2_known_components,
        'composite_gain_vs_same_run_v22':float(h2_composite_gain),
        'min_composite_gain_gate':float(H2_MIN_COMPOSITE_GAIN),
        'keep_candidate':bool(H2_KEEP_CANDIDATE),
    },
    'submission_path':str(SUBMISSION_PATH),
}

SUMMARY_PATH=Path('/kaggle/working/v3_h2_exact_poker_semantics_summary.json')
with open(SUMMARY_PATH,'w') as f:
    json.dump(summary,f,indent=2)

print('\nFINAL V3-H2 EXACT POKER SEMANTICS SUMMARY')
print(json.dumps(summary,indent=2))
print('\n✅ V3-H2 completed its runtime-derived validation gate. Review H2 keep_candidate before leaderboard submission.')


✅ source_schema_checked
✅ metric_weights_70_20_10
✅ development_evidence_membership_validated
✅ pair_outer_train_validation_disjoint
✅ pair_inner_fit_stop_tune_disjoint
✅ pair_inner_subsets_inside_outer_train
✅ pair_outer_validation_not_used_for_risk_early_stop
✅ pair_outer_validation_not_used_for_behavior_early_stop
✅ pair_outer_validation_not_used_for_behavior_rate_tuning
✅ fold_specific_positive_player_weight_sources_outer_train_only
✅ behavior_rate_nested_selected_for_all_folds
✅ behavior_population_weighted_quota_respected
✅ official_exact_ap_used_for_known_pair_reporting
✅ evidence_behavior_probs_not_training_features
✅ family_specific_evidence_rankers
✅ evidence_outer_train_validation_disjoint
✅ evidence_stop_excludes_outer_validation
✅ evidence_probe_train_stop_disjoint
✅ evidence_routing_simulated_oof
✅ evidence_deterministic_tie_break
✅ counterparty_baselines_added
✅ pool_relative_features_added
✅ episodic_features_added
✅ isolation_sequence_motif_added
✅ evidence_binary_rele

# V3-H2 end state

Expected outputs after one Kaggle Run All:

- `/kaggle/working/submission.csv`
- `/kaggle/working/v3_h2_exact_poker_semantics_summary.json`

Protection layers:

1. early H2 dependency/schema/data smoke test before pair/evidence CV;
2. separate H2 pair/evidence OOF gates;
3. final **same-run exact competition-metric composite gate**.

Evaluation semantic features are built only if that final gate passes. If
`h2.keep_candidate` is false, both H2 components revert to V2.2 and the expensive
evaluation semantic stage is skipped.

Recommended Kaggle setting: **Accelerator=None, Internet=Off**.
